In [3]:
%pip install numpy pandas scikit-learn scipy lightgbm jupyter

  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached argon2_cffi_bindings-25.1.0-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl

# Final submission pipeline: `accounting_solver_convex_te_blend_prior`

This notebook keeps only the modelling steps required to reproduce the final submission from the raw competition CSV files.

The final model is built sequentially:

1. Load and preprocess the four source files.
2. Train the required base prediction artifacts: Random Forest, ExtraTrees, base-feature LightGBM, long LightGBM, and target/statistical-encoded LightGBM.
3. Build the verified convex ensemble prior.
4. Apply the subgroup accounting solver and write the final submission CSV.

The original seed is preserved: `RANDOM_STATE = 9890`. The expected final internal OOF MSE is approximately `54.292981`, with solver `equation_weight = 100000` and `lambda_solver = 0.932`.


## 1. Setup

In [1]:
from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

try:
    import lightgbm as lgb
except ImportError as err:
    raise ImportError("Install lightgbm before running this notebook.") from err

try:
    from scipy.optimize import lsq_linear
    HAVE_ACCOUNTING_LSQ_LINEAR = True
except Exception:
    HAVE_ACCOUNTING_LSQ_LINEAR = False

warnings.filterwarnings("ignore", message="X does not have valid feature names")

TARGET_COL = "PERCENT_PROFICIENT"
TARGET = TARGET_COL
ID_COL = "ASSESSMENT_ID"
RANDOM_STATE = 9890
N_SPLITS = 5

np.random.seed(RANDOM_STATE)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("LightGBM version:", lgb.__version__)
print("RANDOM_STATE:", RANDOM_STATE)

LightGBM version: 4.6.0
RANDOM_STATE: 9890


## 2. Data loading and merge

In [2]:
DATA_BASE_URL = "https://michael-weylandt.com/STA9890/competition_data"

scores_training = pd.read_csv(f"{DATA_BASE_URL}/scores_training.csv")
scores_test = pd.read_csv(f"{DATA_BASE_URL}/scores_test.csv")
school_covariates = pd.read_csv(f"{DATA_BASE_URL}/school_covariates.csv")
district_covariates = pd.read_csv(f"{DATA_BASE_URL}/district_covariates.csv")

text_cols = [
    "ASSESSMENT_ID", "SCHOOL", "DISTRICT", "COUNTY",
    "SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "REGION",
]

for df in [scores_training, scores_test, school_covariates, district_covariates]:
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one",
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one",
)

train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one",
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one",
)

train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()
y_train = train_full[TARGET_COL].copy()

X_train = train_full.drop(columns=[TARGET_COL])
X_test = test_full.copy()

print("train_full:", train_full.shape)
print("test_full:", test_full.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

train_full: (144921, 62)
test_full: (48307, 61)
X_train: (144921, 61)
X_test: (48307, 61)


## 3. Base feature matrix

In [3]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

freq_encoding_cols = []

for col in high_cardinality_cols:
    freq_map = X_train_proc[col].value_counts(dropna=False)
    new_col = col + "_freq"

    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)

    freq_encoding_cols.append(new_col)

numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_value_cols = [
    col for col in numeric_cols_extended
    if X_train_proc[col].isna().sum() > 0
]

train_missing_indicators = pd.DataFrame(
    {col + "_missing": X_train_proc[col].isna().astype(int) for col in missing_value_cols},
    index=X_train_proc.index,
)

test_missing_indicators = pd.DataFrame(
    {col + "_missing": X_test_proc[col].isna().astype(int) for col in missing_value_cols},
    index=X_test_proc.index,
)

for col in missing_value_cols:
    median_value = X_train_proc[col].median()
    X_train_proc[col] = X_train_proc[col].fillna(median_value)
    X_test_proc[col] = X_test_proc[col].fillna(median_value)

X_train_proc = pd.concat([X_train_proc, train_missing_indicators], axis=1)
X_test_proc = pd.concat([X_test_proc, test_missing_indicators], axis=1)

X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False,
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False,
)

X_train_proc, X_test_proc = X_train_proc.align(
    X_test_proc,
    join="left",
    axis=1,
    fill_value=0,
)

X_train_proc_model = X_train_proc.drop(columns=[ID_COL]).copy()
X_test_proc_model = X_test_proc.drop(columns=[ID_COL]).copy()

print("Base feature matrix:", X_train_proc_model.shape)
print("Base test matrix:", X_test_proc_model.shape)
print("High-cardinality frequency encodings:", freq_encoding_cols)
print("Low-cardinality one-hot columns:", len(low_cardinality_cols))

Base feature matrix: (144921, 162)
Base test matrix: (48307, 162)
High-cardinality frequency encodings: ['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']
Low-cardinality one-hot columns: 4


## 4. Target/statistical encoding specification

In [4]:
raw_train_te = train_full.reset_index(drop=True).copy()
raw_test_te = test_full.reset_index(drop=True).copy()
y_arr_te = np.asarray(y_train, dtype=np.float32).reshape(-1)

raw_train_te["_N_STUDENTS_BIN_TE"] = pd.cut(
    raw_train_te["N_STUDENTS"].astype(float),
    bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
    labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
).astype("string").fillna("<NA>")

raw_test_te["_N_STUDENTS_BIN_TE"] = pd.cut(
    raw_test_te["N_STUDENTS"].astype(float),
    bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
    labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"],
).astype("string").fillna("<NA>")

te_key_specs = [
    (("ASSESSMENT_NAME",), 20.0, 200.0),
    (("SUBGROUP_NAME",), 20.0, 200.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
    (("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"), 30.0, 250.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),
    (("COUNTY",), 30.0, 250.0),
    (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
    (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
    (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),
    (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT",), 60.0, 400.0),
    (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),
    (("SCHOOL",), 100.0, 600.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
]

print("Target/statistical encoding keys:", len(te_key_specs))
print("Target/statistical encoding features:", len(te_key_specs) * 4)

Target/statistical encoding keys: 17
Target/statistical encoding features: 68


## 5. Shared helpers

In [5]:
import re


def id_array(ids):
    """Convert saved ID objects to a one-dimensional string array."""
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]

    return pd.Series(ids).astype(str).to_numpy()


def append_checkpoint(row_df, path):
    """Append one or more result rows to a CSV checkpoint."""
    write_header = not path.exists()
    row_df.to_csv(path, mode="a", header=write_header, index=False)


def make_group_key(df, cols):
    """Create one grouping key from one or more categorical columns."""
    cols = tuple(cols)

    if len(cols) == 1:
        return (
            df[cols[0]]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
            .reset_index(drop=True)
        )

    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )


def safe_key_name(cols):
    """Make a compact, safe prefix from a tuple of key columns."""
    return "__".join(re.sub(r"[^0-9A-Za-z]+", "_", col).strip("_") for col in cols)


def fit_group_stats(keys, y, n_students, alpha, alpha_n):
    """Learn smoothed group statistics from training rows only."""
    y = np.asarray(y, dtype=np.float64)
    n_students = np.asarray(n_students, dtype=np.float64)

    valid_n = np.isfinite(n_students) & (n_students > 0)

    if not valid_n.all():
        fill_value = np.nanmedian(n_students[valid_n]) if valid_n.any() else 1.0
        n_students = np.where(valid_n, n_students, fill_value)

    y_clipped = np.clip(y, 0.0, 100.0)

    proficient_counts = np.rint((y_clipped / 100.0) * n_students)
    proficient_counts = np.clip(proficient_counts, 0.0, n_students)

    global_mean = float(np.mean(y))
    global_std = float(np.std(y, ddof=0))
    global_weighted_mean = float(
        100.0 * proficient_counts.sum() / max(n_students.sum(), 1.0)
    )

    group_data = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "target": y,
        "n_students": n_students,
        "proficient_count": proficient_counts,
    })

    group_stats = group_data.groupby("key", sort=False).agg(
        count=("target", "size"),
        target_sum=("target", "sum"),
        target_std=("target", "std"),
        n_students_sum=("n_students", "sum"),
        proficient_sum=("proficient_count", "sum"),
    )

    group_stats["mean_smooth"] = (
        group_stats["target_sum"] + alpha * global_mean
    ) / (
        group_stats["count"] + alpha
    )

    group_stats["weighted_mean_smooth"] = 100.0 * (
        group_stats["proficient_sum"] + alpha_n * (global_weighted_mean / 100.0)
    ) / (
        group_stats["n_students_sum"] + alpha_n
    )

    group_stats["target_std"] = group_stats["target_std"].fillna(global_std)
    group_stats["log_count"] = np.log1p(group_stats["count"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_weighted_mean": global_weighted_mean,
        "global_std": global_std,
    }

    return group_stats, defaults


def apply_group_stats(keys, group_stats, defaults, prefix):
    """Apply fitted group statistics; unseen groups use global fallbacks."""
    keys = pd.Series(keys).astype(str).reset_index(drop=True)

    encoded = pd.DataFrame(index=np.arange(len(keys)))

    encoded[f"{prefix}_mean"] = (
        keys.map(group_stats["mean_smooth"])
        .fillna(defaults["global_mean"])
        .astype(np.float32)
    )

    encoded[f"{prefix}_wmean"] = (
        keys.map(group_stats["weighted_mean_smooth"])
        .fillna(defaults["global_weighted_mean"])
        .astype(np.float32)
    )

    encoded[f"{prefix}_log_count"] = (
        keys.map(group_stats["log_count"])
        .fillna(0.0)
        .astype(np.float32)
    )

    encoded[f"{prefix}_std"] = (
        keys.map(group_stats["target_std"])
        .fillna(defaults["global_std"])
        .astype(np.float32)
    )

    return encoded


def take_rows(X, idx):
    """Select rows from either a DataFrame or NumPy array."""
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]

    return X[idx]


def to_float32_matrix(X):
    """Convert a DataFrame or array to a float32 NumPy matrix."""
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)

    return np.asarray(X, dtype=np.float32)


def append_features(X_base, add_df):
    """Append engineered feature columns to a numeric matrix."""
    X_base = to_float32_matrix(X_base)
    X_add = add_df.to_numpy(dtype=np.float32)

    return np.hstack([X_base, X_add])

## 6. Required artifact: Random Forest 500 OOF

In [6]:
start_time = time.perf_counter()

RF_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_oof_500_fold_results.csv"
RF_OOF_PATH = RESULTS_DIR / "oof_rf_500_base.csv"
RF_TEST_FOLD_PATH = RESULTS_DIR / "testpred_rf_500_base_folds.csv"
RF_SUBMISSION_PATH = Path("submission_rf_500_base_oof_foldavg.csv")

for path in [
    RF_OOF_FOLD_RESULTS_PATH,
    RF_OOF_PATH,
    RF_TEST_FOLD_PATH,
    RF_SUBMISSION_PATH,
]:
    if path.exists():
        path.unlink()

RF_OOF_N_SPLITS = 5
RF_OOF_N_ESTIMATORS = 500

rf_oof_params = {
    "n_estimators": RF_OOF_N_ESTIMATORS,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5,
    "bootstrap": True,
    "max_samples": None,
    "oob_score": True,
    "n_jobs": -1,
}

print("RF OOF settings:")
print(rf_oof_params)

kf_rf_oof = KFold(
    n_splits=RF_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

X_rf_all = X_train_proc_model
X_rf_test = X_test_proc_model
y_rf_all = np.asarray(y_train, dtype=float)

rf_oof_pred = np.zeros(len(X_rf_all), dtype=float)
rf_test_pred_folds = np.zeros((len(X_rf_test), RF_OOF_N_SPLITS), dtype=float)

rf_oof_fold_rows = []

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_rf_oof.split(X_rf_all), start=1):
    fold_start = time.perf_counter()

    print("\nRF OOF fold:", fold_id)

    X_fold_train = X_rf_all.iloc[fold_train_idx]
    X_fold_valid = X_rf_all.iloc[fold_valid_idx]

    y_fold_train = y_rf_all[fold_train_idx]
    y_fold_valid = y_rf_all[fold_valid_idx]

    model = RandomForestRegressor(
        **rf_oof_params,
        random_state=RANDOM_STATE + fold_id
    )

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    test_pred = model.predict(X_rf_test)

    rf_oof_pred[fold_valid_idx] = valid_pred
    rf_test_pred_folds[:, fold_id - 1] = test_pred

    oob_pred = model.oob_prediction_
    oob_mask = np.isfinite(oob_pred)

    if oob_mask.sum() > 0:
        oob_mse = mean_squared_error(y_fold_train[oob_mask], oob_pred[oob_mask])
    else:
        oob_mse = np.nan

    row = {
        "model_class": "Random Forest",
        "artifact": "oof_fold_ensemble",
        "fold": fold_id,
        "n_estimators": RF_OOF_N_ESTIMATORS,
        "max_features": 0.5,
        "max_samples": "None",
        "min_samples_leaf": 1,
        "min_samples_split": 2,
        "fold_train_mse": mean_squared_error(y_fold_train, train_pred),
        "fold_oob_mse": oob_mse,
        "fold_valid_mse": mean_squared_error(y_fold_valid, valid_pred),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    rf_oof_fold_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), RF_OOF_FOLD_RESULTS_PATH)

    print("fold_valid_mse:", row["fold_valid_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))

    del model
    gc.collect()

rf_oof_fold_results = pd.DataFrame(rf_oof_fold_rows)

rf_oof_mse = mean_squared_error(y_rf_all, rf_oof_pred)

rf_test_pred_mean_raw = rf_test_pred_folds.mean(axis=1)
rf_test_pred_mean = np.clip(rf_test_pred_mean_raw, 0, 100)

rf_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": train_ids.astype(str),
    "y_true": y_rf_all,
    "rf_500_base_oof_pred": rf_oof_pred,
})

rf_test_fold_df = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
})

for fold_id in range(RF_OOF_N_SPLITS):
    rf_test_fold_df[f"rf_500_base_fold{fold_id + 1}_pred"] = rf_test_pred_folds[:, fold_id]

rf_test_fold_df["rf_500_base_foldavg_pred"] = rf_test_pred_mean

rf_submission_oof = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": rf_test_pred_mean,
})

rf_oof_df.to_csv(RF_OOF_PATH, index=False)
rf_test_fold_df.to_csv(RF_TEST_FOLD_PATH, index=False)
rf_submission_oof.to_csv(RF_SUBMISSION_PATH, index=False)

print("\nOverall RF OOF MSE:", rf_oof_mse)

print("\nSaved files:")
print(RF_OOF_PATH)
print(RF_TEST_FOLD_PATH)
print(RF_SUBMISSION_PATH)

print("\nSubmission shape:", rf_submission_oof.shape)
print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))

RF OOF settings:
{'n_estimators': 500, 'criterion': 'squared_error', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_features': 0.5, 'bootstrap': True, 'max_samples': None, 'oob_score': True, 'n_jobs': -1}

RF OOF fold: 1
fold_valid_mse: 122.88128050280964
elapsed_sec: 441.53

RF OOF fold: 2
fold_valid_mse: 120.14224548830677
elapsed_sec: 626.46

RF OOF fold: 3
fold_valid_mse: 123.54697677953702
elapsed_sec: 485.51

RF OOF fold: 4
fold_valid_mse: 119.76183630285226
elapsed_sec: 479.07

RF OOF fold: 5
fold_valid_mse: 125.30776421592398
elapsed_sec: 581.76

Overall RF OOF MSE: 122.32802447555102

Saved files:
model_results/oof_rf_500_base.csv
model_results/testpred_rf_500_base_folds.csv
submission_rf_500_base_oof_foldavg.csv

Submission shape: (48307, 2)

Elapsed seconds: 2622.28


## 7. Required artifact: ExtraTrees OOF

In [7]:
EXTRATREES_N_JOBS = 1
EXTRATREES_OOF_N_ESTIMATORS = 250
EXTRATREES_OOF_N_SPLITS = 5

EXTRATREES_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "extratrees_oof_fold_results.csv"
EXTRATREES_OOF_SUMMARY_PATH = RESULTS_DIR / "extratrees_oof_summary.csv"
EXTRATREES_OOF_PRED_PATH = RESULTS_DIR / "oof_extratrees_base.csv"
EXTRATREES_TEST_PRED_PATH = RESULTS_DIR / "testpred_extratrees_base_foldavg.csv"
EXTRATREES_SUBMISSION_PATH = Path("submission_extratrees_base_oof_foldavg.csv")

for path in [
    EXTRATREES_OOF_FOLD_RESULTS_PATH,
    EXTRATREES_OOF_SUMMARY_PATH,
    EXTRATREES_OOF_PRED_PATH,
    EXTRATREES_TEST_PRED_PATH,
    EXTRATREES_SUBMISSION_PATH,
]:
    if path.exists():
        path.unlink()

selected_extratrees_config = {
    "config_name": "et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap",
    "n_estimators": EXTRATREES_OOF_N_ESTIMATORS,
    "max_features": 0.50,
    "min_samples_leaf": 1,
    "bootstrap": False,
}

best_extratrees_screen = {
    "config_name": "et_base_180_mf0.5_leaf1_no_bootstrap",
    "holdout_val_mse": 111.62730960485095,
}


def make_extratrees_model(config, random_state):
    """Build the selected ExtraTrees model."""
    return ExtraTreesRegressor(
        n_estimators=config["n_estimators"],
        criterion="squared_error",
        max_depth=None,
        min_samples_leaf=config["min_samples_leaf"],
        min_samples_split=2,
        max_features=config["max_features"],
        bootstrap=config["bootstrap"],
        max_samples=config.get("max_samples", None),
        n_jobs=EXTRATREES_N_JOBS,
        random_state=random_state,
    )

print("Selected ExtraTrees OOF configuration:")
print(selected_extratrees_config)


start_time = time.perf_counter()

X_et_all = X_train_proc_model
X_et_test = X_test_proc_model
y_et_all = np.asarray(y_train, dtype=float)

extratrees_oof_pred = np.full(len(X_et_all), np.nan, dtype=float)
extratrees_test_pred_sum = np.zeros(len(X_et_test), dtype=float)

extratrees_oof_fold_rows = []

kf_extratrees = KFold(
    n_splits=EXTRATREES_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_extratrees.split(X_et_all), start=1):
    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"ExtraTrees OOF fold {fold_id} / {EXTRATREES_OOF_N_SPLITS}")
    print("=" * 80)

    model = make_extratrees_model(
        config=selected_extratrees_config,
        random_state=RANDOM_STATE + 4000 + fold_id
    )

    model.fit(
        X_et_all.iloc[fold_train_idx],
        y_et_all[fold_train_idx]
    )

    valid_pred = model.predict(
        X_et_all.iloc[fold_valid_idx]
    )

    test_pred = model.predict(
        X_et_test
    )

    extratrees_oof_pred[fold_valid_idx] = valid_pred
    extratrees_test_pred_sum += test_pred / EXTRATREES_OOF_N_SPLITS

    row = {
        "model_class": "ExtraTrees",
        "stage": "oof_selected_config",
        "feature_space": "base",
        "fold": fold_id,
        "config_name": selected_extratrees_config["config_name"],
        "n_estimators": selected_extratrees_config["n_estimators"],
        "max_features": selected_extratrees_config["max_features"],
        "min_samples_leaf": selected_extratrees_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_extratrees_config["bootstrap"],
        "max_samples": selected_extratrees_config.get("max_samples", np.nan),
        "fold_valid_mse": mean_squared_error(
            y_et_all[fold_valid_idx],
            valid_pred
        ),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    extratrees_oof_fold_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), EXTRATREES_OOF_FOLD_RESULTS_PATH)

    print("fold_valid_mse:", row["fold_valid_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))
    print(
        "OOF predictions filled:",
        np.isfinite(extratrees_oof_pred).sum(),
        "/",
        len(extratrees_oof_pred)
    )

    del model, valid_pred, test_pred
    gc.collect()

assert np.isfinite(extratrees_oof_pred).all()

extratrees_oof_fold_results = pd.DataFrame(extratrees_oof_fold_rows)
extratrees_oof_fold_results.to_csv(EXTRATREES_OOF_FOLD_RESULTS_PATH, index=False)

extratrees_oof_mse = mean_squared_error(y_et_all, extratrees_oof_pred)

extratrees_oof_pred_clipped = np.clip(extratrees_oof_pred, 0, 100)
extratrees_test_pred_avg = np.clip(extratrees_test_pred_sum, 0, 100)

extratrees_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(train_ids),
    "y_true": y_et_all,
    "extratrees_oof_pred": extratrees_oof_pred,
    "extratrees_oof_pred_clipped": extratrees_oof_pred_clipped,
})

extratrees_test_pred_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "PERCENT_PROFICIENT": extratrees_test_pred_avg,
})

extratrees_oof_df.to_csv(EXTRATREES_OOF_PRED_PATH, index=False)
extratrees_test_pred_df.to_csv(EXTRATREES_TEST_PRED_PATH, index=False)
extratrees_test_pred_df.to_csv(EXTRATREES_SUBMISSION_PATH, index=False)

extratrees_oof_summary = pd.DataFrame([
    {
        "model_key": "ExtraTrees OOF base",
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "screen_holdout_val_mse": float(best_extratrees_screen["holdout_val_mse"]),
        "oof_mse": float(extratrees_oof_mse),
        "fold_valid_mse_mean": float(extratrees_oof_fold_results["fold_valid_mse"].mean()),
        "fold_valid_mse_std": float(extratrees_oof_fold_results["fold_valid_mse"].std()),
        "n_splits": EXTRATREES_OOF_N_SPLITS,
        "n_estimators": EXTRATREES_OOF_N_ESTIMATORS,
        "max_features": selected_extratrees_config["max_features"],
        "min_samples_leaf": selected_extratrees_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_extratrees_config["bootstrap"],
        "max_samples": selected_extratrees_config.get("max_samples", np.nan),
        "n_jobs": EXTRATREES_N_JOBS,
        "submission_path": str(EXTRATREES_SUBMISSION_PATH),
        "elapsed_sec": time.perf_counter() - start_time,
    }
])

extratrees_oof_summary.to_csv(EXTRATREES_OOF_SUMMARY_PATH, index=False)

print("\nSaved files:")
print(EXTRATREES_OOF_FOLD_RESULTS_PATH)
print(EXTRATREES_OOF_SUMMARY_PATH)
print(EXTRATREES_OOF_PRED_PATH)
print(EXTRATREES_TEST_PRED_PATH)
print(EXTRATREES_SUBMISSION_PATH)

Selected ExtraTrees OOF configuration:
{'config_name': 'et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 250, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}

ExtraTrees OOF fold 1 / 5
fold_valid_mse: 111.63933039144383
elapsed_sec: 897.93
OOF predictions filled: 28985 / 144921

ExtraTrees OOF fold 2 / 5
fold_valid_mse: 109.15533429271322
elapsed_sec: 770.97
OOF predictions filled: 57969 / 144921

ExtraTrees OOF fold 3 / 5
fold_valid_mse: 111.87028989470052
elapsed_sec: 502.99
OOF predictions filled: 86953 / 144921

ExtraTrees OOF fold 4 / 5
fold_valid_mse: 108.40540941995584
elapsed_sec: 483.76
OOF predictions filled: 115937 / 144921

ExtraTrees OOF fold 5 / 5
fold_valid_mse: 112.6760763288711
elapsed_sec: 523.0
OOF predictions filled: 144921 / 144921

Saved files:
model_results/extratrees_oof_fold_results.csv
model_results/extratrees_oof_summary.csv
model_results/oof_extratrees_base.csv
model_results/testpred_extratrees_base_foldavg.csv
submission_ex

## 8. Required artifact: selected base-feature LightGBM

In [8]:
LGBM_SELECTED_ARTIFACT = "lgbm_t03_base_5fold_oof"

LGBM_SELECTED_METRICS_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_fold_metrics.csv"
LGBM_SELECTED_OOF_NPY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_oof.npy"
LGBM_SELECTED_TEST_SUM_NPY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_test_pred_sum.npy"

LGBM_SELECTED_OOF_PATH = RESULTS_DIR / f"oof_{LGBM_SELECTED_ARTIFACT}.csv"
LGBM_SELECTED_TEST_PRED_PATH = RESULTS_DIR / f"testpred_{LGBM_SELECTED_ARTIFACT}_foldavg.csv"
LGBM_SELECTED_SUBMISSION_PATH = Path(f"submission_{LGBM_SELECTED_ARTIFACT}_foldavg.csv")

print("Selected LightGBM artifact:", LGBM_SELECTED_ARTIFACT)
print("Metrics path:", LGBM_SELECTED_METRICS_PATH)
print("OOF path:", LGBM_SELECTED_OOF_PATH)
print("Test prediction path:", LGBM_SELECTED_TEST_PRED_PATH)
print("Submission path:", LGBM_SELECTED_SUBMISSION_PATH)

selected_lgbm_12k_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 12000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

selected_lgbm_12k_params

X_lgb_selected_all = np.ascontiguousarray(
    np.asarray(X_train_proc_model, dtype=np.float32)
)

X_lgb_selected_test = np.ascontiguousarray(
    np.asarray(X_test_proc_model, dtype=np.float32)
)

y_lgb_selected_all = np.asarray(
    y_train,
    dtype=np.float32
).ravel()

print("Selected LightGBM training matrix:", X_lgb_selected_all.shape)
print("Selected LightGBM test matrix:", X_lgb_selected_test.shape)
print("Target shape:", y_lgb_selected_all.shape)

if LGBM_SELECTED_METRICS_PATH.exists():
    lgbm_selected_fold_metrics = pd.read_csv(LGBM_SELECTED_METRICS_PATH)
    completed_lgbm_selected_folds = set(
        lgbm_selected_fold_metrics["fold"].astype(int)
    )
else:
    lgbm_selected_fold_metrics = pd.DataFrame()
    completed_lgbm_selected_folds = set()

if LGBM_SELECTED_OOF_NPY_PATH.exists():
    lgbm_selected_oof_pred = np.load(LGBM_SELECTED_OOF_NPY_PATH)
    assert len(lgbm_selected_oof_pred) == len(X_lgb_selected_all)
else:
    lgbm_selected_oof_pred = np.full(
        len(X_lgb_selected_all),
        np.nan,
        dtype=np.float32
    )

if LGBM_SELECTED_TEST_SUM_NPY_PATH.exists():
    lgbm_selected_test_pred_sum = np.load(LGBM_SELECTED_TEST_SUM_NPY_PATH)
    assert len(lgbm_selected_test_pred_sum) == len(X_lgb_selected_test)
else:
    lgbm_selected_test_pred_sum = np.zeros(
        len(X_lgb_selected_test),
        dtype=np.float32
    )

print("Completed selected LightGBM folds:", sorted(completed_lgbm_selected_folds))
print("OOF predictions already filled:", np.isfinite(lgbm_selected_oof_pred).sum())

start_time = time.perf_counter()

kf_lgbm_selected = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_lgbm_selected.split(X_lgb_selected_all), start=1):
    if fold_id in completed_lgbm_selected_folds:
        print(f"\nSkipping completed selected LightGBM fold {fold_id}.")
        continue

    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"Selected LightGBM OOF fold {fold_id} / 5")
    print("=" * 80)

    X_fold_train = X_lgb_selected_all[fold_train_idx]
    X_fold_valid = X_lgb_selected_all[fold_valid_idx]

    y_fold_train = y_lgb_selected_all[fold_train_idx]
    y_fold_valid = y_lgb_selected_all[fold_valid_idx]

    model = lgb.LGBMRegressor(
        **selected_lgbm_12k_params,
        random_state=RANDOM_STATE + fold_id
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=500,
                verbose=True
            ),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iteration = model.best_iteration_

    if best_iteration is None or best_iteration <= 0:
        best_iteration = selected_lgbm_12k_params["n_estimators"]

    valid_pred_raw = model.predict(
        X_fold_valid,
        num_iteration=best_iteration
    )

    valid_pred_clipped = np.clip(
        valid_pred_raw,
        0,
        100
    ).astype(np.float32)

    test_pred_raw = model.predict(
        X_lgb_selected_test,
        num_iteration=best_iteration
    )

    test_pred_clipped = np.clip(
        test_pred_raw,
        0,
        100
    ).astype(np.float32)

    lgbm_selected_oof_pred[fold_valid_idx] = valid_pred_clipped
    lgbm_selected_test_pred_sum += test_pred_clipped / 5.0

    row = {
        "artifact_name": LGBM_SELECTED_ARTIFACT,
        "model_class": "LightGBM",
        "feature_space": "base",
        "fold": fold_id,
        "fold_train_n": len(fold_train_idx),
        "fold_valid_n": len(fold_valid_idx),
        "best_iteration": best_iteration,
        "fold_valid_mse_raw": mean_squared_error(y_fold_valid, valid_pred_raw),
        "fold_valid_mse_clipped": mean_squared_error(y_fold_valid, valid_pred_clipped),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    row.update(selected_lgbm_12k_params)

    lgbm_selected_fold_metrics = pd.concat(
        [
            lgbm_selected_fold_metrics,
            pd.DataFrame([row])
        ],
        axis=0,
        ignore_index=True
    )

    lgbm_selected_fold_metrics.to_csv(
        LGBM_SELECTED_METRICS_PATH,
        index=False
    )

    np.save(
        LGBM_SELECTED_OOF_NPY_PATH,
        lgbm_selected_oof_pred
    )

    np.save(
        LGBM_SELECTED_TEST_SUM_NPY_PATH,
        lgbm_selected_test_pred_sum
    )

    print("best_iteration:", best_iteration)
    print("fold_valid_mse_clipped:", row["fold_valid_mse_clipped"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))
    print("OOF predictions filled:", np.isfinite(lgbm_selected_oof_pred).sum())

    del model
    del X_fold_train, X_fold_valid
    del y_fold_train, y_fold_valid
    del valid_pred_raw, valid_pred_clipped
    del test_pred_raw, test_pred_clipped

    gc.collect()

print("\nSelected LightGBM OOF loop elapsed seconds:", round(time.perf_counter() - start_time, 2))

missing_lgbm_selected_oof = int(np.isnan(lgbm_selected_oof_pred).sum())

print("Missing selected LightGBM OOF predictions:", missing_lgbm_selected_oof)

if missing_lgbm_selected_oof != 0:
    raise RuntimeError(
        "The selected LightGBM OOF artifact is incomplete. Rerun the previous cell to resume."
    )

lgbm_selected_oof_mse = mean_squared_error(
    y_lgb_selected_all,
    lgbm_selected_oof_pred
)

lgbm_selected_test_pred = np.clip(
    lgbm_selected_test_pred_sum,
    0,
    100
).astype(np.float32)

lgbm_selected_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(train_ids),
    "PERCENT_PROFICIENT_TRUE": y_lgb_selected_all,
    f"OOF_{LGBM_SELECTED_ARTIFACT}": lgbm_selected_oof_pred,
})

lgbm_selected_test_pred_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    f"TESTPRED_{LGBM_SELECTED_ARTIFACT}": lgbm_selected_test_pred,
})

lgbm_selected_submission = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "PERCENT_PROFICIENT": lgbm_selected_test_pred,
})

lgbm_selected_summary = pd.DataFrame([
    {
        "model_key": "LightGBM selected 12k OOF base",
        "model_class": "LightGBM",
        "feature_space": "base",
        "artifact_name": LGBM_SELECTED_ARTIFACT,
        "oof_mse_clipped": lgbm_selected_oof_mse,
        "fold_valid_mse_mean": lgbm_selected_fold_metrics["fold_valid_mse_clipped"].mean(),
        "fold_valid_mse_std": lgbm_selected_fold_metrics["fold_valid_mse_clipped"].std(),
        "mean_best_iteration": lgbm_selected_fold_metrics["best_iteration"].mean(),
        "n_splits": 5,
        "n_estimators": selected_lgbm_12k_params["n_estimators"],
        "learning_rate": selected_lgbm_12k_params["learning_rate"],
        "num_leaves": selected_lgbm_12k_params["num_leaves"],
        "min_child_samples": selected_lgbm_12k_params["min_child_samples"],
        "subsample": selected_lgbm_12k_params["subsample"],
        "colsample_bytree": selected_lgbm_12k_params["colsample_bytree"],
        "reg_alpha": selected_lgbm_12k_params["reg_alpha"],
        "reg_lambda": selected_lgbm_12k_params["reg_lambda"],
        "submission_path": str(LGBM_SELECTED_SUBMISSION_PATH),
    }
])

lgbm_selected_oof_df.to_csv(
    LGBM_SELECTED_OOF_PATH,
    index=False
)

lgbm_selected_test_pred_df.to_csv(
    LGBM_SELECTED_TEST_PRED_PATH,
    index=False
)

lgbm_selected_submission.to_csv(
    LGBM_SELECTED_SUBMISSION_PATH,
    index=False
)

LGBM_SELECTED_SUMMARY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_summary.csv"

lgbm_selected_summary.to_csv(
    LGBM_SELECTED_SUMMARY_PATH,
    index=False
)

print("Selected LightGBM OOF MSE:", lgbm_selected_oof_mse)

print("\nSaved files:")
print(LGBM_SELECTED_METRICS_PATH)
print(LGBM_SELECTED_OOF_PATH)
print(LGBM_SELECTED_TEST_PRED_PATH)
print(LGBM_SELECTED_SUBMISSION_PATH)
print(LGBM_SELECTED_SUMMARY_PATH)

Selected LightGBM artifact: lgbm_t03_base_5fold_oof
Metrics path: model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
OOF path: model_results/oof_lgbm_t03_base_5fold_oof.csv
Test prediction path: model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv
Submission path: submission_lgbm_t03_base_5fold_oof_foldavg.csv
Selected LightGBM training matrix: (144921, 162)
Selected LightGBM test matrix: (48307, 162)
Target shape: (144921,)
Completed selected LightGBM folds: [1, 2, 3, 4, 5]
OOF predictions already filled: 144921

Skipping completed selected LightGBM fold 1.

Skipping completed selected LightGBM fold 2.

Skipping completed selected LightGBM fold 3.

Skipping completed selected LightGBM fold 4.

Skipping completed selected LightGBM fold 5.

Selected LightGBM OOF loop elapsed seconds: 0.02
Missing selected LightGBM OOF predictions: 0
Selected LightGBM OOF MSE: 98.77940368652344

Saved files:
model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
model_results/oof_lgbm_t03_base_

## 9. Required artifacts: long base-feature LightGBM models

In [9]:
LONG_LGBM_N_SPLITS = 5

X_long_lgbm = np.ascontiguousarray(
    np.asarray(X_train_proc_model, dtype=np.float32)
)

X_long_lgbm_test = np.ascontiguousarray(
    np.asarray(X_test_proc_model, dtype=np.float32)
)

y_long_lgbm = np.asarray(
    y_train,
    dtype=np.float32
).ravel()

print("Long LightGBM training matrix:", X_long_lgbm.shape)
print("Long LightGBM test matrix:", X_long_lgbm_test.shape)
print("Target shape:", y_long_lgbm.shape)

long_lgbm_base_params = {
    "objective": "regression",
    "metric": "l2",
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

long_lgbm_jobs = [
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long80k_lr03",
        "learning_rate": 0.03,
        "n_estimators": 80000,
        "early_stopping_rounds": 2500,
        "log_period": 2000,
    },
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long100k_lr02",
        "learning_rate": 0.02,
        "n_estimators": 100000,
        "early_stopping_rounds": 3000,
        "log_period": 2500,
    },
]

pd.DataFrame(long_lgbm_jobs)

def run_long_lgbm_artifact(job):
    artifact_name = job["artifact_name"]

    metrics_path = RESULTS_DIR / f"{artifact_name}_fold_metrics.csv"
    oof_npy_path = RESULTS_DIR / f"{artifact_name}_oof.npy"

    oof_csv_path = RESULTS_DIR / f"oof_{artifact_name}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact_name}_foldavg.csv"
    submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

    params = long_lgbm_base_params.copy()
    params.update({
        "n_estimators": job["n_estimators"],
        "learning_rate": job["learning_rate"],
    })

    if metrics_path.exists():
        fold_metrics = pd.read_csv(metrics_path)
        completed_folds = set(fold_metrics["fold"].astype(int))
    else:
        fold_metrics = pd.DataFrame()
        completed_folds = set()

    if oof_npy_path.exists():
        oof_pred = np.load(oof_npy_path)
        assert len(oof_pred) == len(X_long_lgbm)
    else:
        oof_pred = np.full(
            len(X_long_lgbm),
            np.nan,
            dtype=np.float32
        )

    print("\n" + "#" * 90)
    print("Long LightGBM artifact:", artifact_name)
    print("#" * 90)

    print("\nParameters:")
    for key, value in params.items():
        print(f"{key}: {value}")

    print("\nCompleted folds:", sorted(completed_folds))
    print("OOF predictions filled:", int(np.isfinite(oof_pred).sum()))

    kf = KFold(
        n_splits=LONG_LGBM_N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    artifact_start = time.perf_counter()

    for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf.split(X_long_lgbm), start=1):
        fold_test_path = RESULTS_DIR / f"{artifact_name}_fold{fold_id}_test_pred.npy"

        fold_oof_done = np.isfinite(oof_pred[fold_valid_idx]).all()
        fold_test_done = fold_test_path.exists()
        fold_metric_done = fold_id in completed_folds

        if fold_oof_done and fold_test_done and fold_metric_done:
            print(f"\nSkipping completed fold {fold_id}.")
            continue

        if fold_oof_done or fold_test_done or fold_metric_done:
            print(f"\nFold {fold_id} has partial checkpoint state. Rerunning it cleanly.")

            if len(fold_metrics) > 0:
                fold_metrics = fold_metrics[
                    fold_metrics["fold"].astype(int) != fold_id
                ].copy()

                fold_metrics.to_csv(
                    metrics_path,
                    index=False
                )

            if fold_test_path.exists():
                fold_test_path.unlink()

            oof_pred[fold_valid_idx] = np.nan
            np.save(oof_npy_path, oof_pred)

        print("\n" + "=" * 80)
        print(f"{artifact_name}: fold {fold_id} / {LONG_LGBM_N_SPLITS}")
        print("=" * 80)

        fold_start = time.perf_counter()

        X_fold_train = X_long_lgbm[fold_train_idx]
        X_fold_valid = X_long_lgbm[fold_valid_idx]

        y_fold_train = y_long_lgbm[fold_train_idx]
        y_fold_valid = y_long_lgbm[fold_valid_idx]

        model = lgb.LGBMRegressor(
            **params,
            random_state=RANDOM_STATE + fold_id
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            eval_set=[(X_fold_valid, y_fold_valid)],
            eval_metric="l2",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=job["early_stopping_rounds"],
                    verbose=True
                ),
                lgb.log_evaluation(
                    period=job["log_period"]
                ),
            ],
        )

        best_iteration = model.best_iteration_

        if best_iteration is None or best_iteration <= 0:
            best_iteration = job["n_estimators"]

        valid_pred_raw = model.predict(
            X_fold_valid,
            num_iteration=best_iteration
        )

        valid_pred_clipped = np.clip(
            valid_pred_raw,
            0,
            100
        ).astype(np.float32)

        test_pred_raw = model.predict(
            X_long_lgbm_test,
            num_iteration=best_iteration
        )

        test_pred_clipped = np.clip(
            test_pred_raw,
            0,
            100
        ).astype(np.float32)

        oof_pred[fold_valid_idx] = valid_pred_clipped

        np.save(
            oof_npy_path,
            oof_pred
        )

        np.save(
            fold_test_path,
            test_pred_clipped
        )

        row = {
            "artifact_name": artifact_name,
            "model_class": "LightGBM",
            "feature_space": "base",
            "fold": fold_id,
            "fold_train_n": len(fold_train_idx),
            "fold_valid_n": len(fold_valid_idx),
            "best_iteration": best_iteration,
            "fold_valid_mse_raw": mean_squared_error(y_fold_valid, valid_pred_raw),
            "fold_valid_mse_clipped": mean_squared_error(y_fold_valid, valid_pred_clipped),
            "elapsed_sec": time.perf_counter() - fold_start,
        }

        row.update(params)

        fold_metrics = pd.concat(
            [
                fold_metrics,
                pd.DataFrame([row])
            ],
            axis=0,
            ignore_index=True
        )

        fold_metrics = (
            fold_metrics
            .sort_values("fold")
            .reset_index(drop=True)
        )

        fold_metrics.to_csv(
            metrics_path,
            index=False
        )

        print("best_iteration:", best_iteration)
        print("fold_valid_mse_clipped:", row["fold_valid_mse_clipped"])
        print("elapsed_sec:", round(row["elapsed_sec"], 2))
        print("OOF predictions filled:", int(np.isfinite(oof_pred).sum()))

        del model
        del X_fold_train, X_fold_valid
        del y_fold_train, y_fold_valid
        del valid_pred_raw, valid_pred_clipped
        del test_pred_raw, test_pred_clipped

        gc.collect()

    missing_oof = int(np.isnan(oof_pred).sum())

    fold_test_arrays = []

    for fold_id in range(1, LONG_LGBM_N_SPLITS + 1):
        fold_test_path = RESULTS_DIR / f"{artifact_name}_fold{fold_id}_test_pred.npy"

        if not fold_test_path.exists():
            print("Missing fold test prediction:", fold_test_path)
            continue

        fold_test_arrays.append(
            np.load(fold_test_path).astype(np.float32)
        )

    if missing_oof != 0 or len(fold_test_arrays) != LONG_LGBM_N_SPLITS:
        print("\nArtifact is incomplete. Rerun this same cell to resume.")
        print("Missing OOF predictions:", missing_oof)
        print("Fold test files found:", len(fold_test_arrays))

        return {
            "artifact_name": artifact_name,
            "completed": False,
            "oof_mse_clipped": np.nan,
            "metrics_path": metrics_path,
            "oof_csv_path": oof_csv_path,
            "testpred_csv_path": testpred_csv_path,
            "submission_path": submission_path,
        }

    oof_mse_clipped = mean_squared_error(
        y_long_lgbm,
        oof_pred
    )

    test_pred_foldavg = np.clip(
        np.mean(np.vstack(fold_test_arrays), axis=0),
        0,
        100
    ).astype(np.float32)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(train_ids),
        "PERCENT_PROFICIENT_TRUE": y_long_lgbm,
        f"OOF_{artifact_name}": oof_pred,
    })

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(test_ids),
        f"TESTPRED_{artifact_name}": test_pred_foldavg,
    })

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(test_ids),
        "PERCENT_PROFICIENT": test_pred_foldavg,
    })

    oof_df.to_csv(
        oof_csv_path,
        index=False
    )

    testpred_df.to_csv(
        testpred_csv_path,
        index=False
    )

    submission_df.to_csv(
        submission_path,
        index=False
    )

    artifact_elapsed = time.perf_counter() - artifact_start

    print("\nCompleted artifact:", artifact_name)
    print("OOF MSE clipped:", oof_mse_clipped)

    print("\nSaved:")
    print(metrics_path)
    print(oof_csv_path)
    print(testpred_csv_path)
    print(submission_path)

    print("\nArtifact elapsed seconds:", round(artifact_elapsed, 2))

    return {
        "artifact_name": artifact_name,
        "completed": True,
        "oof_mse_clipped": float(oof_mse_clipped),
        "metrics_path": metrics_path,
        "oof_csv_path": oof_csv_path,
        "testpred_csv_path": testpred_csv_path,
        "submission_path": submission_path,
    }

long80k_result = run_long_lgbm_artifact(
    long_lgbm_jobs[0]
)

long80k_result

long100k_result = run_long_lgbm_artifact(
    long_lgbm_jobs[1]
)

long100k_result

Long LightGBM training matrix: (144921, 162)
Long LightGBM test matrix: (48307, 162)
Target shape: (144921,)

##########################################################################################
Long LightGBM artifact: lgbm_t03_base_5fold_oof_long80k_lr03
##########################################################################################

Parameters:
objective: regression
metric: l2
n_jobs: 1
verbosity: -1
force_col_wise: True
num_leaves: 95
min_child_samples: 60
subsample: 0.85
subsample_freq: 1
colsample_bytree: 0.9
reg_alpha: 0.0
reg_lambda: 5.0
max_depth: -1
n_estimators: 80000
learning_rate: 0.03

Completed folds: [1, 2, 3, 4, 5]
OOF predictions filled: 144921

Skipping completed fold 1.

Skipping completed fold 2.

Skipping completed fold 3.

Skipping completed fold 4.

Skipping completed fold 5.

Completed artifact: lgbm_t03_base_5fold_oof_long80k_lr03
OOF MSE clipped: 94.16348266601562

Saved:
model_results/lgbm_t03_base_5fold_oof_long80k_lr03_fold_metrics.csv
mode

{'artifact_name': 'lgbm_t03_base_5fold_oof_long100k_lr02',
 'completed': True,
 'oof_mse_clipped': 93.7262191772461,
 'metrics_path': PosixPath('model_results/lgbm_t03_base_5fold_oof_long100k_lr02_fold_metrics.csv'),
 'oof_csv_path': PosixPath('model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv'),
 'testpred_csv_path': PosixPath('model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv'),
 'submission_path': PosixPath('submission_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv')}

## 10. Required artifact: target/statistical-encoded LightGBM

In [10]:
TE_LGBM_ARTIFACT = "lgbm_te_base_5fold_oof_v1"
TE_LGBM_N_SPLITS = 5

TE_LGBM_METRICS_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold_metrics.csv"
TE_LGBM_OOF_RAW_NPY_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_oof_raw.npy"
TE_LGBM_OOF_CLIPPED_NPY_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_oof_clipped.npy"
TE_LGBM_OOF_PATH = RESULTS_DIR / f"oof_{TE_LGBM_ARTIFACT}.csv"
TE_LGBM_TEST_PATH = RESULTS_DIR / f"testpred_{TE_LGBM_ARTIFACT}_foldavg.csv"
TE_LGBM_SUBMISSION_PATH = Path(f"submission_{TE_LGBM_ARTIFACT}_foldavg.csv")

y_te_lgbm_all = np.asarray(y_train, dtype=np.float32).reshape(-1)

te_lgbm_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "n_estimators": 20000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

n_train_te_lgbm = X_train_proc_model.shape[0]
n_test_te_lgbm = X_test_proc_model.shape[0]

te_outer_kf = KFold(
    n_splits=TE_LGBM_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

te_outer_fold_indices = list(te_outer_kf.split(np.arange(n_train_te_lgbm)))

if TE_LGBM_OOF_RAW_NPY_PATH.exists():
    te_lgbm_oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)
else:
    te_lgbm_oof_raw = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

if TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
    te_lgbm_oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)
else:
    te_lgbm_oof_clipped = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

if TE_LGBM_METRICS_PATH.exists():
    te_lgbm_fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)
else:
    te_lgbm_fold_metrics = pd.DataFrame()

TE_OOF_PREFLIGHT_PASS = bool(
    len(te_key_specs) > 0
    and X_train_proc_model.shape[0] == len(y_te_lgbm_all)
    and X_test_proc_model.shape[0] == len(raw_test_te)
    and len(test_ids) == len(raw_test_te)
    and X_train_proc_model.shape[1] + len(te_key_specs) * 4 == 230
)

print("TE_OOF_PREFLIGHT_PASS:", TE_OOF_PREFLIGHT_PASS)
print("TE artifact:", TE_LGBM_ARTIFACT)
print("Base features:", X_train_proc_model.shape[1])
print("TE features:", len(te_key_specs) * 4)
print("Total features:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)

if not TE_OOF_PREFLIGHT_PASS:
    raise ValueError("Target/statistical-encoded LightGBM preflight failed.")


def build_te_oof_and_apply_many(
    raw_fit,
    y_fit,
    raw_apply_dict,
    key_specs,
    n_splits=5,
    random_state=9890,
):
    """
    Build leakage-safe target/statistical encoding features for one outer fold.

    The function is used once per outer fold.

    For the outer-training rows:
    - target/statistical encoding features are built out-of-fold.

    For the outer-validation and test rows:
    - mappings are learned only from the outer-training rows.
    """
    raw_fit = raw_fit.reset_index(drop=True)

    raw_apply_dict = {
        name: frame.reset_index(drop=True)
        for name, frame in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = raw_fit["N_STUDENTS"].astype(float).to_numpy()

    inner_kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    oof_feature_blocks = []
    apply_feature_blocks = {
        name: []
        for name in raw_apply_dict
    }

    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        fit_keys = make_group_key(raw_fit, cols)

        apply_keys = {
            name: make_group_key(frame, cols)
            for name, frame in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_values = np.zeros(
            (len(raw_fit), len(feature_cols)),
            dtype=np.float32,
        )

        for inner_train_idx, inner_valid_idx in inner_kf.split(np.arange(len(raw_fit))):
            fold_stats, fold_defaults = fit_group_stats(
                fit_keys.iloc[inner_train_idx],
                y_fit[inner_train_idx],
                n_fit[inner_train_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            encoded_inner_valid = apply_group_stats(
                fit_keys.iloc[inner_valid_idx],
                fold_stats,
                fold_defaults,
                prefix,
            )

            oof_values[inner_valid_idx, :] = encoded_inner_valid[feature_cols].to_numpy(
                dtype=np.float32
            )

        full_stats, full_defaults = fit_group_stats(
            fit_keys,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_feature_blocks.append(
            pd.DataFrame(oof_values, columns=feature_cols)
        )

        for name, keys in apply_keys.items():
            encoded_apply = apply_group_stats(
                keys,
                full_stats,
                full_defaults,
                prefix,
            )

            apply_feature_blocks[name].append(
                encoded_apply[feature_cols]
            )

        fit_counts = fit_keys.value_counts(dropna=False)

        summary_row = {
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        }

        fit_group_set = set(fit_counts.index)

        for name, keys in apply_keys.items():
            summary_row[f"{name}_groups"] = int(keys.nunique(dropna=False))
            summary_row[f"{name}_row_coverage"] = float(keys.isin(fit_group_set).mean())

        summary_rows.append(summary_row)

    oof_features = pd.concat(oof_feature_blocks, axis=1)

    apply_features = {
        name: pd.concat(blocks, axis=1)
        for name, blocks in apply_feature_blocks.items()
    }

    key_summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, key_summary

def run_te_lgbm_fold(fold_id):
    """
    Train one fold of the target/statistical-encoded LightGBM artifact.

    This function is called separately for folds 1 through 5 so each fold has its own notebook cell.
    """
    if not TE_OOF_PREFLIGHT_PASS:
        raise ValueError("TE_OOF_PREFLIGHT_PASS is False. Re-run the Part 6 preflight cell.")

    if fold_id < 1 or fold_id > TE_LGBM_N_SPLITS:
        raise ValueError(f"fold_id must be between 1 and {TE_LGBM_N_SPLITS}.")

    fold_train_idx, fold_valid_idx = te_outer_fold_indices[fold_id - 1]
    fold_test_path = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"

    if TE_LGBM_OOF_RAW_NPY_PATH.exists():
        oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)
    else:
        oof_raw = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

    if TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
        oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)
    else:
        oof_clipped = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

    if TE_LGBM_METRICS_PATH.exists():
        fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)
    else:
        fold_metrics = pd.DataFrame()

    fold_already_complete = (
        "fold" in fold_metrics.columns
        and fold_id in set(fold_metrics["fold"].astype(int))
        and fold_test_path.exists()
        and np.isfinite(oof_clipped[fold_valid_idx]).all()
    )

    if fold_already_complete:
        print(f"TE LightGBM fold {fold_id} is already complete. Skipping training.")
        return (
            fold_metrics
            .loc[fold_metrics["fold"].astype(int) == fold_id]
            .iloc[0]
            .to_dict()
        )

    print("\n" + "=" * 80)
    print(f"Training TE LightGBM fold {fold_id} / {TE_LGBM_N_SPLITS}")
    print("=" * 80)

    fold_start = time.perf_counter()

    raw_fit_fold = raw_train_te.iloc[fold_train_idx].reset_index(drop=True)
    raw_valid_fold = raw_train_te.iloc[fold_valid_idx].reset_index(drop=True)

    y_fit_fold = y_te_lgbm_all[fold_train_idx]
    y_valid_fold = y_te_lgbm_all[fold_valid_idx]

    print("Building leakage-safe TE features for this fold...")

    te_fit_fold, te_apply_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit_fold,
        raw_apply_dict={
            "valid": raw_valid_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=TE_LGBM_N_SPLITS,
        random_state=RANDOM_STATE + 100 * fold_id,
    )

    finite_te_ok = (
        np.isfinite(te_fit_fold.to_numpy(dtype=np.float32)).all()
        and np.isfinite(te_apply_fold["valid"].to_numpy(dtype=np.float32)).all()
        and np.isfinite(te_apply_fold["test"].to_numpy(dtype=np.float32)).all()
    )

    if not finite_te_ok:
        raise ValueError(f"Non-finite TE features detected in fold {fold_id}.")

    X_fit_base_fold = to_float32_matrix(
        take_rows(X_train_proc_model, fold_train_idx)
    )

    X_valid_base_fold = to_float32_matrix(
        take_rows(X_train_proc_model, fold_valid_idx)
    )

    X_test_base_fold = to_float32_matrix(X_test_proc_model)

    X_fit_fold = append_features(X_fit_base_fold, te_fit_fold)
    X_valid_fold = append_features(X_valid_base_fold, te_apply_fold["valid"])
    X_test_fold = append_features(X_test_base_fold, te_apply_fold["test"])

    print("Fold train shape:", X_fit_fold.shape)
    print("Fold valid shape:", X_valid_fold.shape)
    print("Fold test shape:", X_test_fold.shape)

    model = lgb.LGBMRegressor(**te_lgbm_params)

    print("Training LightGBM...")

    model.fit(
        X_fit_fold,
        y_fit_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iteration = int(model.best_iteration_ or te_lgbm_params["n_estimators"])

    fit_pred_raw = model.predict(
        X_fit_fold,
        num_iteration=best_iteration
    )

    valid_pred_raw = model.predict(
        X_valid_fold,
        num_iteration=best_iteration
    )

    test_pred_raw = model.predict(
        X_test_fold,
        num_iteration=best_iteration
    )

    fit_pred_clipped = np.clip(fit_pred_raw, 0, 100).astype(np.float32)
    valid_pred_clipped = np.clip(valid_pred_raw, 0, 100).astype(np.float32)
    test_pred_clipped = np.clip(test_pred_raw, 0, 100).astype(np.float32)

    oof_raw[fold_valid_idx] = valid_pred_raw.astype(np.float32)
    oof_clipped[fold_valid_idx] = valid_pred_clipped

    np.save(TE_LGBM_OOF_RAW_NPY_PATH, oof_raw)
    np.save(TE_LGBM_OOF_CLIPPED_NPY_PATH, oof_clipped)
    np.save(fold_test_path, test_pred_clipped)

    elapsed_seconds = time.perf_counter() - fold_start

    fold_row = {
        "artifact": TE_LGBM_ARTIFACT,
        "fold": fold_id,
        "train_rows": int(len(fold_train_idx)),
        "valid_rows": int(len(fold_valid_idx)),
        "n_features": int(X_fit_fold.shape[1]),
        "best_iteration": best_iteration,
        "train_mse_raw": mean_squared_error(y_fit_fold, fit_pred_raw),
        "train_mse_clipped": mean_squared_error(y_fit_fold, fit_pred_clipped),
        "valid_mse_raw": mean_squared_error(y_valid_fold, valid_pred_raw),
        "valid_mse_clipped": mean_squared_error(y_valid_fold, valid_pred_clipped),
        "pred_valid_mean": float(valid_pred_clipped.mean()),
        "pred_valid_std": float(valid_pred_clipped.std()),
        "pred_valid_min": float(valid_pred_clipped.min()),
        "pred_valid_max": float(valid_pred_clipped.max()),
        "elapsed_seconds": elapsed_seconds,
    }

    if len(fold_metrics) > 0 and "fold" in fold_metrics.columns:
        fold_metrics = fold_metrics[
            fold_metrics["fold"].astype(int) != fold_id
        ].copy()

    fold_metrics = pd.concat(
        [
            fold_metrics,
            pd.DataFrame([fold_row]),
        ],
        ignore_index=True,
    )

    fold_metrics = (
        fold_metrics
        .sort_values("fold")
        .reset_index(drop=True)
    )

    fold_metrics.to_csv(
        TE_LGBM_METRICS_PATH,
        index=False
    )

    del model
    del raw_fit_fold, raw_valid_fold
    del te_fit_fold, te_apply_fold, te_key_summary_fold
    del X_fit_base_fold, X_valid_base_fold, X_test_base_fold
    del X_fit_fold, X_valid_fold, X_test_fold
    del fit_pred_raw, valid_pred_raw, test_pred_raw
    del fit_pred_clipped, valid_pred_clipped, test_pred_clipped

    gc.collect()

    return fold_row


te_lgbm_fold_rows = []

for fold_id in range(1, TE_LGBM_N_SPLITS + 1):
    te_lgbm_fold_rows.append(run_te_lgbm_fold(fold_id))

pd.DataFrame(te_lgbm_fold_rows)


if not TE_LGBM_OOF_RAW_NPY_PATH.exists():
    raise FileNotFoundError(f"Missing raw OOF checkpoint: {TE_LGBM_OOF_RAW_NPY_PATH}")

if not TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
    raise FileNotFoundError(f"Missing clipped OOF checkpoint: {TE_LGBM_OOF_CLIPPED_NPY_PATH}")

if not TE_LGBM_METRICS_PATH.exists():
    raise FileNotFoundError(f"Missing fold metrics file: {TE_LGBM_METRICS_PATH}")

te_lgbm_oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)
te_lgbm_oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)
te_lgbm_fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)

missing_fold_test_paths = [
    RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"
    for fold_id in range(1, TE_LGBM_N_SPLITS + 1)
    if not (RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy").exists()
]

TE_OOF_RUN_PASS = bool(
    np.isfinite(te_lgbm_oof_raw).all()
    and np.isfinite(te_lgbm_oof_clipped).all()
    and len(missing_fold_test_paths) == 0
    and te_lgbm_fold_metrics["fold"].nunique() == TE_LGBM_N_SPLITS
)

print("TE_OOF_RUN_PASS:", TE_OOF_RUN_PASS)

if missing_fold_test_paths:
    print("Missing fold test prediction files:")
    for path in missing_fold_test_paths:
        print("-", path)

if not TE_OOF_RUN_PASS:
    raise ValueError("The TE LightGBM OOF artifact is incomplete. Finish all five fold cells before finalizing.")

te_lgbm_oof_mse_raw = mean_squared_error(
    y_te_lgbm_all,
    te_lgbm_oof_raw
)

te_lgbm_oof_mse_clipped = mean_squared_error(
    y_te_lgbm_all,
    te_lgbm_oof_clipped
)

te_lgbm_fold_test_preds = []

for fold_id in range(1, TE_LGBM_N_SPLITS + 1):
    fold_test_path = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"
    te_lgbm_fold_test_preds.append(
        np.load(fold_test_path).astype(np.float32)
    )

te_lgbm_test_pred = np.mean(
    np.vstack(te_lgbm_fold_test_preds),
    axis=0
)

te_lgbm_test_pred = np.clip(
    te_lgbm_test_pred,
    0,
    100
).astype(np.float32)

te_lgbm_oof_df = pd.DataFrame({
    "row_index": np.arange(n_train_te_lgbm),
    TARGET_COL: y_te_lgbm_all,
    "pred_raw": te_lgbm_oof_raw,
    "pred_clipped": te_lgbm_oof_clipped,
})

te_lgbm_test_df = pd.DataFrame({
    ID_COL: id_array(test_ids),
    TARGET_COL: te_lgbm_test_pred,
})

te_lgbm_submission = te_lgbm_test_df[
    [ID_COL, TARGET_COL]
].copy()

te_lgbm_oof_df.to_csv(
    TE_LGBM_OOF_PATH,
    index=False
)

te_lgbm_test_df.to_csv(
    TE_LGBM_TEST_PATH,
    index=False
)

te_lgbm_submission.to_csv(
    TE_LGBM_SUBMISSION_PATH,
    index=False
)

te_lgbm_artifact_summary = pd.DataFrame([
    {
        "artifact": TE_LGBM_ARTIFACT,
        "feature_space": "base + target/statistical encoding",
        "base_features": X_train_proc_model.shape[1],
        "target_encoding_features": len(te_key_specs) * 4,
        "total_features": X_train_proc_model.shape[1] + len(te_key_specs) * 4,
        "folds": TE_LGBM_N_SPLITS,
        "mean_fold_valid_mse": te_lgbm_fold_metrics["valid_mse_clipped"].mean(),
        "overall_oof_mse_raw": te_lgbm_oof_mse_raw,
        "overall_oof_mse_clipped": te_lgbm_oof_mse_clipped,
        "test_pred_mean": te_lgbm_test_pred.mean(),
        "test_pred_std": te_lgbm_test_pred.std(),
    }
])

print("Saved files:")
print("-", TE_LGBM_OOF_PATH)
print("-", TE_LGBM_TEST_PATH)
print("-", TE_LGBM_SUBMISSION_PATH)

print("\nOOF MSE raw:", te_lgbm_oof_mse_raw)
print("OOF MSE clipped:", te_lgbm_oof_mse_clipped)

te_lgbm_artifact_summary

TE_OOF_PREFLIGHT_PASS: True
TE artifact: lgbm_te_base_5fold_oof_v1
Base features: 162
TE features: 68
Total features: 230
TE LightGBM fold 1 is already complete. Skipping training.
TE LightGBM fold 2 is already complete. Skipping training.
TE LightGBM fold 3 is already complete. Skipping training.
TE LightGBM fold 4 is already complete. Skipping training.
TE LightGBM fold 5 is already complete. Skipping training.
TE_OOF_RUN_PASS: True
Saved files:
- model_results/oof_lgbm_te_base_5fold_oof_v1.csv
- model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
- submission_lgbm_te_base_5fold_oof_v1_foldavg.csv

OOF MSE raw: 83.01273345947266
OOF MSE clipped: 82.90760803222656


,artifact,feature_space,base_features,target_encoding_features,total_features,folds,mean_fold_valid_mse,overall_oof_mse_raw,overall_oof_mse_clipped,test_pred_mean,test_pred_std
0,lgbm_te_base_5fold_oof_v1,base + target/statistical encoding,162,68,230,5,82.907608,83.012733,82.907608,54.500694,24.778822


## 11. Convex ensemble prior

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

required_part6d_objects = [
    "y_train",
    "train_ids",
    "test_ids",
]

missing_part6d_objects = [
    name for name in required_part6d_objects
    if name not in globals()
]

if missing_part6d_objects:
    raise ValueError(f"Missing required objects for Part 6D: {missing_part6d_objects}")

y_safe_te_blend = np.asarray(y_train, dtype=float).reshape(-1)
train_id_reference = pd.Series(train_ids).astype(str).to_numpy()
test_id_reference = pd.Series(test_ids).astype(str).to_numpy()

rf_oof_path = Path(globals().get("RF_OOF_PATH", RESULTS_DIR / "oof_rf_500_base.csv"))
rf_test_path = Path(globals().get("RF_TEST_FOLD_PATH", RESULTS_DIR / "testpred_rf_500_base_folds.csv"))

extratrees_oof_path = Path(globals().get("EXTRATREES_OOF_PRED_PATH", RESULTS_DIR / "oof_extratrees_base.csv"))
extratrees_test_path = Path(globals().get("EXTRATREES_TEST_PRED_PATH", RESULTS_DIR / "testpred_extratrees_base_foldavg.csv"))

lgbm_selected_artifact = globals().get("LGBM_SELECTED_ARTIFACT", "lgbm_t03_base_5fold_oof")
lgbm_selected_oof_path = Path(
    globals().get(
        "LGBM_SELECTED_OOF_PATH",
        RESULTS_DIR / f"oof_{lgbm_selected_artifact}.csv",
    )
)
lgbm_selected_test_path = Path(
    globals().get(
        "LGBM_SELECTED_TEST_PRED_PATH",
        RESULTS_DIR / f"testpred_{lgbm_selected_artifact}_foldavg.csv",
    )
)

long80k_artifact = "lgbm_t03_base_5fold_oof_long80k_lr03"
long100k_artifact = "lgbm_t03_base_5fold_oof_long100k_lr02"

long80k_oof_path = Path(
    long80k_result["oof_csv_path"]
    if "long80k_result" in globals()
    else RESULTS_DIR / f"oof_{long80k_artifact}.csv"
)

long80k_test_path = Path(
    long80k_result["testpred_csv_path"]
    if "long80k_result" in globals()
    else RESULTS_DIR / f"testpred_{long80k_artifact}_foldavg.csv"
)

long100k_oof_path = Path(
    long100k_result["oof_csv_path"]
    if "long100k_result" in globals()
    else RESULTS_DIR / f"oof_{long100k_artifact}.csv"
)

long100k_test_path = Path(
    long100k_result["testpred_csv_path"]
    if "long100k_result" in globals()
    else RESULTS_DIR / f"testpred_{long100k_artifact}_foldavg.csv"
)

te_lgbm_oof_path = RESULTS_DIR / "oof_lgbm_te_base_5fold_oof_v1.csv"
te_lgbm_test_path = RESULTS_DIR / "testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv"

safe_te_component_specs = [
    {
        "name": "rf500",
        "source_section": "Part 4",
        "display_name": "Random Forest 500",
        "oof_path": rf_oof_path,
        "test_path": rf_test_path,
        "oof_pred_col": "rf_500_base_oof_pred",
        "test_pred_col": "rf_500_base_foldavg_pred",
        "target_col": "y_true",
    },
    {
        "name": "extratrees",
        "source_section": "Part 4",
        "display_name": "ExtraTrees",
        "oof_path": extratrees_oof_path,
        "test_path": extratrees_test_path,
        "oof_pred_col": "extratrees_oof_pred",
        "test_pred_col": TARGET_COL,
        "target_col": "y_true",
    },
    {
        "name": "lgbm_12k",
        "source_section": "Part 5",
        "display_name": "LightGBM selected 12k",
        "oof_path": lgbm_selected_oof_path,
        "test_path": lgbm_selected_test_path,
        "oof_pred_col": f"OOF_{lgbm_selected_artifact}",
        "test_pred_col": f"TESTPRED_{lgbm_selected_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_80k_lr03",
        "source_section": "Part 5",
        "display_name": "LightGBM long 80k",
        "oof_path": long80k_oof_path,
        "test_path": long80k_test_path,
        "oof_pred_col": f"OOF_{long80k_artifact}",
        "test_pred_col": f"TESTPRED_{long80k_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_100k_lr02",
        "source_section": "Part 5",
        "display_name": "LightGBM long 100k",
        "oof_path": long100k_oof_path,
        "test_path": long100k_test_path,
        "oof_pred_col": f"OOF_{long100k_artifact}",
        "test_pred_col": f"TESTPRED_{long100k_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_te_v1",
        "source_section": "Part 6",
        "display_name": "Target/statistical-encoded LightGBM",
        "oof_path": te_lgbm_oof_path,
        "test_path": te_lgbm_test_path,
        "oof_pred_col": "pred_clipped",
        "test_pred_col": TARGET_COL,
        "target_col": TARGET_COL,
    },
]

component_file_check = []

for component in safe_te_component_specs:
    component_file_check.append({
        "component": component["name"],
        "source_section": component["source_section"],
        "display_name": component["display_name"],
        "oof_path": str(component["oof_path"]),
        "oof_exists": component["oof_path"].exists(),
        "test_path": str(component["test_path"]),
        "test_exists": component["test_path"].exists(),
    })

component_file_check = pd.DataFrame(component_file_check)

PART6D_COMPONENT_FILES_READY = bool(
    component_file_check["oof_exists"].all()
    and component_file_check["test_exists"].all()
)

print("PART6D_COMPONENT_FILES_READY:", PART6D_COMPONENT_FILES_READY)

if not PART6D_COMPONENT_FILES_READY:
    raise FileNotFoundError(
        "One or more required blend component artifacts are missing. "
        "Random Forest and ExtraTrees should come from Part 4; "
        "base and long LightGBM artifacts should come from Part 5; "
        "the TE LightGBM artifact should come from Part 6C."
    )

component_file_check

SAFE_TE_BLEND_NAME = "blend_te_verified_noresid_weighted"

SAFE_TE_BLEND_OOF_PATH = RESULTS_DIR / f"oof_{SAFE_TE_BLEND_NAME}.csv"
SAFE_TE_BLEND_TEST_PATH = RESULTS_DIR / f"testpred_{SAFE_TE_BLEND_NAME}.csv"
SAFE_TE_BLEND_SUBMISSION_PATH = Path(f"submission_{SAFE_TE_BLEND_NAME}.csv")

SAFE_TE_BLEND_SCREEN_PATH = RESULTS_DIR / "blend_te_verified_noresid_weight_screen.csv"
SAFE_TE_BLEND_WEIGHTS_PATH = RESULTS_DIR / "blend_te_verified_noresid_best_weights.csv"

safe_te_blend_component_specs = [
    {
        "name": "rf500",
        "display_name": "Random Forest 500",
        "source_section": "Part 4",
        "oof_path": rf_oof_path,
        "test_path": rf_test_path,
        "oof_pred_cols": ["rf_500_base_oof_pred"],
        "test_pred_cols": ["rf_500_base_foldavg_pred"],
        "target_col": "y_true",
    },
    {
        "name": "extratrees_safe",
        "display_name": "ExtraTrees",
        "source_section": "Part 4",
        "oof_path": extratrees_oof_path,
        "test_path": extratrees_test_path,
        "oof_pred_cols": ["extratrees_oof_pred", "extratrees_oof_pred_clipped", "oof_pred_clipped", "oof_pred"],
        "test_pred_cols": [TARGET_COL, "PERCENT_PROFICIENT", "extratrees_test_pred"],
        "target_col": "y_true",
    },
    {
        "name": "lgbm_12k",
        "display_name": "LightGBM selected 12k",
        "source_section": "Part 5",
        "oof_path": lgbm_selected_oof_path,
        "test_path": lgbm_selected_test_path,
        "oof_pred_cols": [f"OOF_{lgbm_selected_artifact}"],
        "test_pred_cols": [f"TESTPRED_{lgbm_selected_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_long80k",
        "display_name": "LightGBM long 80k",
        "source_section": "Part 5",
        "oof_path": long80k_oof_path,
        "test_path": long80k_test_path,
        "oof_pred_cols": [f"OOF_{long80k_artifact}"],
        "test_pred_cols": [f"TESTPRED_{long80k_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_long100k",
        "display_name": "LightGBM long 100k",
        "source_section": "Part 5",
        "oof_path": long100k_oof_path,
        "test_path": long100k_test_path,
        "oof_pred_cols": [f"OOF_{long100k_artifact}"],
        "test_pred_cols": [f"TESTPRED_{long100k_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_te_v1",
        "display_name": "Target/statistical-encoded LightGBM",
        "source_section": "Part 6",
        "oof_path": te_lgbm_oof_path,
        "test_path": te_lgbm_test_path,
        "oof_pred_cols": ["pred_clipped"],
        "test_pred_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
        "target_col": TARGET_COL,
    },
]


def first_present_column(df, candidate_cols, file_path, component_name):
    """
    Return the first expected prediction column present in a saved artifact.

    This is used for every blend component because some artifacts have cleaned
    names while the archive has original-compatible names.
    """
    for col in candidate_cols:
        if col in df.columns:
            return col

    raise ValueError(
        f"No expected prediction column found for {component_name} in {file_path}. "
        f"Expected one of {candidate_cols}. Observed columns: {list(df.columns)}"
    )


def load_aligned_oof_prediction(component):
    """
    Load one OOF prediction artifact and align it to the current training row order.
    """
    path = Path(component["oof_path"])
    df = pd.read_csv(path)

    pred_col = first_present_column(
        df=df,
        candidate_cols=component["oof_pred_cols"],
        file_path=path,
        component_name=component["name"],
    )

    aligned = df.copy()

    if "row_index" in aligned.columns:
        aligned = (
            aligned
            .sort_values("row_index")
            .reset_index(drop=True)
        )

        expected_row_index = np.arange(len(aligned))

        if not np.array_equal(aligned["row_index"].to_numpy(), expected_row_index):
            raise ValueError(f"OOF row_index is not consecutive for {component['name']}.")

    elif ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in OOF file for {component['name']}.")

        train_order = pd.DataFrame({
            ID_COL: train_id_reference,
            "_train_order": np.arange(len(train_id_reference)),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = train_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"OOF predictions could not be aligned for {component['name']}.")

        aligned = (
            aligned
            .sort_values("_train_order")
            .drop(columns=["_train_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

    if len(aligned) != len(y_safe_te_blend):
        raise ValueError(
            f"OOF row count mismatch for {component['name']}: "
            f"{len(aligned)} rows vs {len(y_safe_te_blend)} expected."
        )

    target_col = component["target_col"]

    if target_col in aligned.columns:
        saved_target = aligned[target_col].to_numpy(dtype=float)

        if not np.allclose(saved_target, y_safe_te_blend):
            raise ValueError(f"Saved target values do not align for {component['name']}.")

    pred = aligned[pred_col].to_numpy(dtype=float)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions found for {component['name']}.")

    return np.clip(pred, 0, 100), pred_col


def load_aligned_test_prediction(component):
    """
    Load one test prediction artifact and align it to the current test row order.
    """
    path = Path(component["test_path"])
    df = pd.read_csv(path)

    pred_col = first_present_column(
        df=df,
        candidate_cols=component["test_pred_cols"],
        file_path=path,
        component_name=component["name"],
    )

    aligned = df.copy()

    if ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in test file for {component['name']}.")

        test_order = pd.DataFrame({
            ID_COL: test_id_reference,
            "_test_order": np.arange(len(test_id_reference)),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = test_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"Test predictions could not be aligned for {component['name']}.")

        aligned = (
            aligned
            .sort_values("_test_order")
            .drop(columns=["_test_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

    if len(aligned) != len(test_id_reference):
        raise ValueError(
            f"Test row count mismatch for {component['name']}: "
            f"{len(aligned)} rows vs {len(test_id_reference)} expected."
        )

    pred = aligned[pred_col].to_numpy(dtype=float)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions found for {component['name']}.")

    return np.clip(pred, 0, 100), pred_col

safe_te_component_names = []
safe_te_component_display_names = []
safe_te_oof_predictions = []
safe_te_test_predictions = []
safe_te_component_rows = []

for component in safe_te_blend_component_specs:
    oof_pred, oof_pred_col = load_aligned_oof_prediction(component)
    test_pred, test_pred_col = load_aligned_test_prediction(component)

    component_oof_mse = mean_squared_error(
        y_safe_te_blend,
        oof_pred,
    )

    safe_te_component_names.append(component["name"])
    safe_te_component_display_names.append(component["display_name"])
    safe_te_oof_predictions.append(oof_pred)
    safe_te_test_predictions.append(test_pred)

    safe_te_component_rows.append({
        "component": component["name"],
        "display_name": component["display_name"],
        "source_section": component["source_section"],
        "oof_mse": component_oof_mse,
        "oof_file": str(component["oof_path"]),
        "oof_column": oof_pred_col,
        "test_file": str(component["test_path"]),
        "test_column": test_pred_col,
        "oof_mean": oof_pred.mean(),
        "oof_std": oof_pred.std(),
        "test_mean": test_pred.mean(),
        "test_std": test_pred.std(),
    })

safe_te_oof_matrix = np.column_stack(safe_te_oof_predictions).astype(np.float64)
safe_te_test_matrix = np.column_stack(safe_te_test_predictions).astype(np.float64)

safe_te_component_summary = (
    pd.DataFrame(safe_te_component_rows)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

SAFE_TE_BLEND_PREFLIGHT_PASS = bool(
    safe_te_component_names == [
        "rf500",
        "extratrees_safe",
        "lgbm_12k",
        "lgbm_long80k",
        "lgbm_long100k",
        "lgbm_te_v1",
    ]
    and safe_te_oof_matrix.shape == (len(y_safe_te_blend), 6)
    and safe_te_test_matrix.shape == (len(test_id_reference), 6)
    and np.isfinite(safe_te_oof_matrix).all()
    and np.isfinite(safe_te_test_matrix).all()
)

print("SAFE_TE_BLEND_PREFLIGHT_PASS:", SAFE_TE_BLEND_PREFLIGHT_PASS)
print("Component order:", safe_te_component_names)
print("OOF matrix shape:", safe_te_oof_matrix.shape)
print("Test matrix shape:", safe_te_test_matrix.shape)

if not SAFE_TE_BLEND_PREFLIGHT_PASS:
    raise ValueError("Safe TE blend component loading failed.")

safe_te_component_summary

n_safe_te_blend, k_safe_te_blend = safe_te_oof_matrix.shape

safe_te_gram = (safe_te_oof_matrix.T @ safe_te_oof_matrix) / n_safe_te_blend
safe_te_target_cross = (safe_te_oof_matrix.T @ y_safe_te_blend) / n_safe_te_blend
safe_te_target_square = float((y_safe_te_blend @ y_safe_te_blend) / n_safe_te_blend)


def safe_te_mse_for_weights(weights):
    """
    Compute OOF MSE for one convex weight vector.
    """
    weights = np.asarray(weights, dtype=np.float64)

    return float(
        weights @ safe_te_gram @ weights
        - 2.0 * (weights @ safe_te_target_cross)
        + safe_te_target_square
    )


def safe_te_mse_for_weight_matrix(weight_matrix):
    """
    Compute OOF MSE for many convex weight vectors.
    """
    weight_matrix = np.asarray(weight_matrix, dtype=np.float64)

    return (
        np.einsum("ij,jk,ik->i", weight_matrix, safe_te_gram, weight_matrix)
        - 2.0 * (weight_matrix @ safe_te_target_cross)
        + safe_te_target_square
    )


safe_te_blend_candidates = []


def add_safe_te_candidate(label, weights):
    """
    Add one normalized nonnegative blend candidate.
    """
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.maximum(weights, 0)

    if weights.sum() <= 0:
        return

    weights = weights / weights.sum()

    safe_te_blend_candidates.append({
        "label": label,
        "mse": safe_te_mse_for_weights(weights),
        "weights": weights,
    })


for component_idx, component_name in enumerate(safe_te_component_names):
    weights = np.zeros(k_safe_te_blend, dtype=np.float64)
    weights[component_idx] = 1.0

    add_safe_te_candidate(
        label=f"pure_{component_name}",
        weights=weights,
    )


pair_grid = np.linspace(0.0, 1.0, 1001)

for i in range(k_safe_te_blend):
    for j in range(i + 1, k_safe_te_blend):
        weight_matrix = np.zeros(
            (len(pair_grid), k_safe_te_blend),
            dtype=np.float64,
        )

        weight_matrix[:, i] = pair_grid
        weight_matrix[:, j] = 1.0 - pair_grid

        pair_mses = safe_te_mse_for_weight_matrix(weight_matrix)
        best_pair_idx = int(np.argmin(pair_mses))

        add_safe_te_candidate(
            label=f"pair_{safe_te_component_names[i]}__{safe_te_component_names[j]}",
            weights=weight_matrix[best_pair_idx],
        )


safe_te_rng = np.random.default_rng(RANDOM_STATE + 2809)

uniform_weights = safe_te_rng.dirichlet(
    np.ones(k_safe_te_blend),
    size=50000,
)

uniform_mses = safe_te_mse_for_weight_matrix(uniform_weights)

add_safe_te_candidate(
    label="random_dirichlet_uniform_50k",
    weights=uniform_weights[int(np.argmin(uniform_mses))],
)


component_mses = np.array([
    mean_squared_error(y_safe_te_blend, safe_te_oof_matrix[:, i])
    for i in range(k_safe_te_blend)
])

te_component_idx = safe_te_component_names.index("lgbm_te_v1")
component_order_by_mse = np.argsort(component_mses)

te_focused_alpha = np.ones(k_safe_te_blend) * 0.25
te_focused_alpha[te_component_idx] = 10.0

for idx in component_order_by_mse[:min(3, k_safe_te_blend)]:
    te_focused_alpha[idx] = max(te_focused_alpha[idx], 2.5)

te_focused_weights = safe_te_rng.dirichlet(
    te_focused_alpha,
    size=75000,
)

te_focused_mses = safe_te_mse_for_weight_matrix(te_focused_weights)

add_safe_te_candidate(
    label="random_dirichlet_te_focused_75k",
    weights=te_focused_weights[int(np.argmin(te_focused_mses))],
)


top_focused_alpha = np.ones(k_safe_te_blend) * 0.20

for rank, idx in enumerate(component_order_by_mse[:min(4, k_safe_te_blend)]):
    top_focused_alpha[idx] = 6.0 / (rank + 1)

top_focused_weights = safe_te_rng.dirichlet(
    top_focused_alpha,
    size=75000,
)

top_focused_mses = safe_te_mse_for_weight_matrix(top_focused_weights)

add_safe_te_candidate(
    label="random_dirichlet_top_focused_75k",
    weights=top_focused_weights[int(np.argmin(top_focused_mses))],
)


safe_te_blend_screen_rows = []

for candidate in safe_te_blend_candidates:
    row = {
        "label": candidate["label"],
        "mse": candidate["mse"],
    }

    for component_name, weight in zip(safe_te_component_names, candidate["weights"]):
        row[f"w_{component_name}"] = weight

    safe_te_blend_screen_rows.append(row)

safe_te_blend_screen = (
    pd.DataFrame(safe_te_blend_screen_rows)
    .sort_values("mse")
    .reset_index(drop=True)
)

best_safe_te_blend = safe_te_blend_candidates[
    int(np.argmin([candidate["mse"] for candidate in safe_te_blend_candidates]))
]

best_safe_te_weights = best_safe_te_blend["weights"]

safe_te_blend_oof_pred = np.clip(
    safe_te_oof_matrix @ best_safe_te_weights,
    0,
    100,
)

safe_te_blend_test_pred = np.clip(
    safe_te_test_matrix @ best_safe_te_weights,
    0,
    100,
)

safe_te_blend_oof_mse = mean_squared_error(
    y_safe_te_blend,
    safe_te_blend_oof_pred,
)

pure_te_oof_mse_for_blend = mean_squared_error(
    y_safe_te_blend,
    safe_te_oof_matrix[:, te_component_idx],
)

safe_te_blend_gain_vs_te = pure_te_oof_mse_for_blend - safe_te_blend_oof_mse

safe_te_blend_weight_table = (
    pd.DataFrame({
        "component": safe_te_component_names,
        "display_name": safe_te_component_display_names,
        "weight": best_safe_te_weights,
        "component_oof_mse": component_mses,
    })
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

print("Best candidate:", best_safe_te_blend["label"])
print("Safe TE blend OOF MSE:", safe_te_blend_oof_mse)
print("Pure TE LightGBM OOF MSE:", pure_te_oof_mse_for_blend)
print("Gain vs pure TE LightGBM:", safe_te_blend_gain_vs_te)

print("\nBest blend weights:")
print(safe_te_blend_weight_table.to_string(index=False))

print("\nTop blend candidates:")
safe_te_blend_screen.head(20)

safe_te_blend_oof_df = pd.DataFrame({
    "row_index": np.arange(len(y_safe_te_blend)),
    TARGET_COL: y_safe_te_blend,
    "pred_clipped": safe_te_blend_oof_pred,
})

safe_te_blend_test_df = pd.DataFrame({
    ID_COL: test_id_reference,
    TARGET_COL: safe_te_blend_test_pred,
})

safe_te_blend_submission = safe_te_blend_test_df[
    [ID_COL, TARGET_COL]
].copy()

safe_te_blend_oof_df.to_csv(
    SAFE_TE_BLEND_OOF_PATH,
    index=False,
)

safe_te_blend_test_df.to_csv(
    SAFE_TE_BLEND_TEST_PATH,
    index=False,
)

safe_te_blend_submission.to_csv(
    SAFE_TE_BLEND_SUBMISSION_PATH,
    index=False,
)

safe_te_blend_screen.to_csv(
    SAFE_TE_BLEND_SCREEN_PATH,
    index=False,
)

safe_te_blend_weight_table.to_csv(
    SAFE_TE_BLEND_WEIGHTS_PATH,
    index=False,
)

SAFE_TE_BLEND_PASS = True
SAFE_TE_BLEND_SUBMISSION_WORTHY = bool(safe_te_blend_gain_vs_te >= 1.0)

print("SAFE_TE_BLEND_PASS:", SAFE_TE_BLEND_PASS)
print("SAFE_TE_BLEND_SUBMISSION_WORTHY:", SAFE_TE_BLEND_SUBMISSION_WORTHY)

print("\nSaved files:")
print("-", SAFE_TE_BLEND_OOF_PATH)
print("-", SAFE_TE_BLEND_TEST_PATH)
print("-", SAFE_TE_BLEND_SUBMISSION_PATH)
print("-", SAFE_TE_BLEND_SCREEN_PATH)
print("-", SAFE_TE_BLEND_WEIGHTS_PATH)

print("\nSubmission prediction summary:")
pd.Series(safe_te_blend_test_pred).describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

PART6D_COMPONENT_FILES_READY: True
SAFE_TE_BLEND_PREFLIGHT_PASS: True
Component order: ['rf500', 'extratrees_safe', 'lgbm_12k', 'lgbm_long80k', 'lgbm_long100k', 'lgbm_te_v1']
OOF matrix shape: (144921, 6)
Test matrix shape: (48307, 6)
Best candidate: random_dirichlet_te_focused_75k
Safe TE blend OOF MSE: 78.07151037235208
Pure TE LightGBM OOF MSE: 82.90761370632671
Gain vs pure TE LightGBM: 4.836103333974634

Best blend weights:
      component                        display_name   weight  component_oof_mse
     lgbm_te_v1 Target/statistical-encoded LightGBM 0.624592          82.907614
  lgbm_long100k                  LightGBM long 100k 0.207687          93.726214
   lgbm_long80k                   LightGBM long 80k 0.127595          94.163485
extratrees_safe                          ExtraTrees 0.040027         110.749294
       lgbm_12k               LightGBM selected 12k 0.000082          98.779406
          rf500                   Random Forest 500 0.000017         122.328024

Top bl

count    48307.000000
mean        54.338382
std         24.551656
min          0.201202
1%           7.931219
5%          16.555257
25%         34.791862
50%         52.371320
75%         74.537515
95%         95.197915
99%         99.362563
max         99.999194
dtype: float64

## 12. Accounting solver and final submission

In [12]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

try:
    from scipy.optimize import lsq_linear
    HAVE_ACCOUNTING_LSQ_LINEAR = True
except Exception:
    HAVE_ACCOUNTING_LSQ_LINEAR = False

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ACCOUNTING_N_SPLITS = 5

required_accounting_objects = [
    "raw_train_te",
    "raw_test_te",
    "y_train",
]

missing_accounting_objects = [
    name for name in required_accounting_objects
    if name not in globals()
]

if missing_accounting_objects:
    raise ValueError(f"Missing required accounting objects: {missing_accounting_objects}")

required_accounting_columns = [
    ID_COL,
    "SCHOOL",
    "ASSESSMENT_NAME",
    "SUBGROUP_NAME",
    "N_STUDENTS",
]

missing_train_accounting_cols = [
    col for col in required_accounting_columns
    if col not in raw_train_te.columns
]

missing_test_accounting_cols = [
    col for col in required_accounting_columns
    if col not in raw_test_te.columns
]

if missing_train_accounting_cols:
    raise ValueError(f"raw_train_te is missing columns: {missing_train_accounting_cols}")

if missing_test_accounting_cols:
    raise ValueError(f"raw_test_te is missing columns: {missing_test_accounting_cols}")

y_accounting = np.asarray(y_train, dtype=np.float64).reshape(-1)

if len(raw_train_te) != len(y_accounting):
    raise ValueError("raw_train_te and y_train have different row counts.")

n_train_accounting = len(raw_train_te)
n_test_accounting = len(raw_test_te)

train_id_reference = pd.Series(raw_train_te[ID_COL]).astype(str).to_numpy()
test_id_reference = pd.Series(raw_test_te[ID_COL]).astype(str).to_numpy()
test_ids_output = raw_test_te[ID_COL].to_numpy()


def clean_accounting_string(values):
    return pd.Series(values).astype("string").fillna("<NA>").astype(str)


def accounting_numeric_array(values):
    return (
        pd.to_numeric(values, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )


train_accounting_df = pd.DataFrame({
    "row_index": np.arange(n_train_accounting),
    ID_COL: raw_train_te[ID_COL].to_numpy(),
    "SCHOOL": clean_accounting_string(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_accounting_string(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_accounting_string(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": accounting_numeric_array(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_accounting,
})

test_accounting_df = pd.DataFrame({
    "row_index": np.arange(n_test_accounting),
    ID_COL: raw_test_te[ID_COL].to_numpy(),
    "SCHOOL": clean_accounting_string(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_accounting_string(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_accounting_string(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": accounting_numeric_array(raw_test_te["N_STUDENTS"]),
})

for frame in [train_accounting_df, test_accounting_df]:
    frame["N_STUDENTS"] = np.where(
        np.isfinite(frame["N_STUDENTS"]) & (frame["N_STUDENTS"] > 0),
        frame["N_STUDENTS"],
        np.nan,
    )

    frame["group_key"] = (
        frame["SCHOOL"].astype(str)
        + "||"
        + frame["ASSESSMENT_NAME"].astype(str)
    )

train_accounting_df["prof_count"] = np.rint(
    np.clip(train_accounting_df[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train_accounting_df["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train_accounting_df["prof_count"] = np.clip(
    train_accounting_df["prof_count"],
    0,
    train_accounting_df["N_STUDENTS"],
)

candidate_accounting_identities = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

subgroups_seen = set(train_accounting_df["SUBGROUP_NAME"]).union(
    set(test_accounting_df["SUBGROUP_NAME"])
)

accounting_identities = [
    identity for identity in candidate_accounting_identities
    if all(subgroup in subgroups_seen for subgroup in identity)
]

if not accounting_identities:
    raise ValueError("No usable subgroup accounting identities were found.")

subgroup_count_summary = pd.concat(
    [
        train_accounting_df["SUBGROUP_NAME"].value_counts().rename("train_rows"),
        test_accounting_df["SUBGROUP_NAME"].value_counts().rename("test_rows"),
    ],
    axis=1,
).fillna(0).astype(int)

print("Have scipy.optimize.lsq_linear:", HAVE_ACCOUNTING_LSQ_LINEAR)
print("Training rows:", n_train_accounting)
print("Test rows:", n_test_accounting)

print("\nAccounting identities used:")
for all_s, a_s, b_s in accounting_identities:
    print(f"{all_s} = {a_s} + {b_s}")

subgroup_count_summary

def first_available_prediction_column(df, candidate_cols, path, label):
    """
    Return the first prediction column present in a saved prediction file.
    """
    for col in candidate_cols:
        if col in df.columns:
            return col

    numeric_cols = [
        col for col in df.columns
        if col not in ["row_index", ID_COL, "fold"]
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if TARGET_COL in numeric_cols and len(numeric_cols) > 1:
        numeric_cols = [col for col in numeric_cols if col != TARGET_COL]

    if numeric_cols:
        return numeric_cols[0]

    raise ValueError(
        f"No usable prediction column found for {label} in {path}. "
        f"Observed columns: {list(df.columns)}"
    )


def load_accounting_oof_prior(path, candidate_cols, label):
    """
    Load and align one OOF prior prediction vector.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    pred_col = first_available_prediction_column(df, candidate_cols, path, label)

    aligned = df.copy()

    if "row_index" in aligned.columns:
        aligned = aligned.sort_values("row_index").reset_index(drop=True)

        if len(aligned) != n_train_accounting:
            raise ValueError(f"OOF row count mismatch for {label}.")

        expected_row_index = np.arange(n_train_accounting)

        if not np.array_equal(aligned["row_index"].to_numpy(), expected_row_index):
            raise ValueError(f"OOF row_index is not consecutive for {label}.")

    elif ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in OOF file for {label}.")

        train_order = pd.DataFrame({
            ID_COL: train_id_reference,
            "_train_order": np.arange(n_train_accounting),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = train_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"OOF predictions could not be aligned for {label}.")

        aligned = (
            aligned
            .sort_values("_train_order")
            .drop(columns=["_train_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

        if len(aligned) != n_train_accounting:
            raise ValueError(f"OOF row count mismatch for {label}.")

    if TARGET_COL in aligned.columns and pred_col != TARGET_COL:
        saved_target = pd.to_numeric(aligned[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)

        if np.isfinite(saved_target).all() and not np.allclose(saved_target, y_accounting):
            raise ValueError(f"OOF target alignment failed for {label}.")

    pred = pd.to_numeric(aligned[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF prior predictions found for {label}.")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_accounting_test_prior(path, candidate_cols, label):
    """
    Load and align one test prior prediction vector.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    pred_col = first_available_prediction_column(df, candidate_cols, path, label)

    aligned = df.copy()

    if ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in test file for {label}.")

        test_order = pd.DataFrame({
            ID_COL: test_id_reference,
            "_test_order": np.arange(n_test_accounting),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = test_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"Test predictions could not be aligned for {label}.")

        aligned = (
            aligned
            .sort_values("_test_order")
            .drop(columns=["_test_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

        if len(aligned) != n_test_accounting:
            raise ValueError(f"Test row count mismatch for {label}.")

    pred = pd.to_numeric(aligned[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test prior predictions found for {label}.")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


accounting_prior_specs = [
    {
        "name": "te_lgbm_prior",
        "display_name": "Target/statistical-encoded LightGBM prior",
        "oof_path": RESULTS_DIR / "oof_lgbm_te_base_5fold_oof_v1.csv",
        "test_path": RESULTS_DIR / "testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv",
        "oof_candidate_cols": ["pred_clipped"],
        "test_candidate_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
    },
    {
        "name": "convex_te_blend_prior",
        "display_name": "Verified convex ensemble prior",
        "oof_path": RESULTS_DIR / "oof_blend_te_verified_noresid_weighted.csv",
        "test_path": RESULTS_DIR / "testpred_blend_te_verified_noresid_weighted.csv",
        "oof_candidate_cols": ["pred_clipped"],
        "test_candidate_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
    },
]

accounting_prior_data = {}
accounting_prior_rows = []

for spec in accounting_prior_specs:
    oof_pred, oof_col = load_accounting_oof_prior(
        path=spec["oof_path"],
        candidate_cols=spec["oof_candidate_cols"],
        label=spec["name"],
    )

    test_pred, test_col = load_accounting_test_prior(
        path=spec["test_path"],
        candidate_cols=spec["test_candidate_cols"],
        label=spec["name"],
    )

    prior_oof_mse = mean_squared_error(y_accounting, oof_pred)

    accounting_prior_data[spec["name"]] = {
        "display_name": spec["display_name"],
        "oof_pred": oof_pred,
        "test_pred": test_pred,
        "oof_path": spec["oof_path"],
        "test_path": spec["test_path"],
        "oof_column": oof_col,
        "test_column": test_col,
        "base_oof_mse": prior_oof_mse,
    }

    accounting_prior_rows.append({
        "prior": spec["name"],
        "display_name": spec["display_name"],
        "base_oof_mse": prior_oof_mse,
        "oof_file": str(spec["oof_path"]),
        "oof_column": oof_col,
        "test_file": str(spec["test_path"]),
        "test_column": test_col,
        "test_pred_mean": test_pred.mean(),
        "test_pred_std": test_pred.std(),
    })

accounting_prior_summary = (
    pd.DataFrame(accounting_prior_rows)
    .sort_values("base_oof_mse")
    .reset_index(drop=True)
)

ACCOUNTING_PRIORS_READY = bool(
    set(accounting_prior_data) == {"te_lgbm_prior", "convex_te_blend_prior"}
    and all(len(data["oof_pred"]) == n_train_accounting for data in accounting_prior_data.values())
    and all(len(data["test_pred"]) == n_test_accounting for data in accounting_prior_data.values())
)

print("ACCOUNTING_PRIORS_READY:", ACCOUNTING_PRIORS_READY)

if not ACCOUNTING_PRIORS_READY:
    raise ValueError("Accounting prior loading failed.")

accounting_prior_summary

def accounting_n_tolerance(n_students):
    """
    Tolerance for checking whether subgroup N_STUDENTS totals are compatible.
    """
    if not np.isfinite(n_students):
        return 1.0

    return max(1.0, 0.02 * float(n_students))


def build_accounting_known_lookup(df, row_indices):
    """
    Build known subgroup proficient-count lookup from selected training rows.
    """
    subset = df.iloc[row_indices]

    lookup = {}

    for group_key, group_df in subset.groupby("group_key", sort=False):
        group_lookup = {}

        for subgroup, subgroup_df in group_df.groupby("SUBGROUP_NAME", sort=False):
            n_values = subgroup_df["N_STUDENTS"].to_numpy(dtype=np.float64)
            k_values = subgroup_df["prof_count"].to_numpy(dtype=np.float64)

            good = (
                np.isfinite(n_values)
                & (n_values > 0)
                & np.isfinite(k_values)
            )

            if not good.any():
                continue

            n_mean = float(np.mean(n_values[good]))
            k_mean = float(np.mean(k_values[good]))

            group_lookup[str(subgroup)] = {
                "n": n_mean,
                "k": float(np.clip(k_mean, 0, n_mean)),
            }

        if group_lookup:
            lookup[str(group_key)] = group_lookup

    return lookup


def solve_one_accounting_group(
    query_group_df,
    known_group,
    prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    """
    Solve one SCHOOL x ASSESSMENT_NAME accounting system.
    """
    n_query = len(query_group_df)

    pred_pct = np.full(n_query, np.nan, dtype=np.float32)
    touched = np.zeros(n_query, dtype=bool)
    eq_count_used = 0

    subgroup_to_var = {}
    var_to_local = []
    var_n = []
    var_prior_count = []

    for local_idx, row in query_group_df.iterrows():
        subgroup = str(row["SUBGROUP_NAME"])
        n_students = float(row["N_STUDENTS"])
        prior_value = float(prior_pct[local_idx])

        if not np.isfinite(n_students) or n_students <= 0:
            continue

        if not np.isfinite(prior_value):
            continue

        if subgroup in subgroup_to_var:
            continue

        var_idx = len(var_to_local)

        subgroup_to_var[subgroup] = var_idx
        var_to_local.append(local_idx)
        var_n.append(n_students)
        var_prior_count.append(float(np.clip(prior_value, 0, 100) / 100.0 * n_students))

    n_vars = len(var_to_local)

    if n_vars == 0:
        return pred_pct, touched, eq_count_used

    def subgroup_present(subgroup):
        return (
            subgroup in subgroup_to_var
            or (
                known_group is not None
                and subgroup in known_group
            )
        )

    def subgroup_n(subgroup):
        if subgroup in subgroup_to_var:
            return var_n[subgroup_to_var[subgroup]]

        return known_group[subgroup]["n"]

    def subgroup_known_count(subgroup):
        return known_group[subgroup]["k"]

    a_rows = []
    b_values = []

    sqrt_prior = float(np.sqrt(prior_weight))
    sqrt_equation = float(np.sqrt(equation_weight))

    for var_idx in range(n_vars):
        row = np.zeros(n_vars, dtype=np.float64)
        row[var_idx] = sqrt_prior

        a_rows.append(row)
        b_values.append(sqrt_prior * var_prior_count[var_idx])

    touched_vars = set()

    for all_s, a_s, b_s in identities:
        if not (
            subgroup_present(all_s)
            and subgroup_present(a_s)
            and subgroup_present(b_s)
        ):
            continue

        n_all = subgroup_n(all_s)
        n_a = subgroup_n(a_s)
        n_b = subgroup_n(b_s)

        if not (
            np.isfinite(n_all)
            and np.isfinite(n_a)
            and np.isfinite(n_b)
        ):
            continue

        if abs(n_all - (n_a + n_b)) > accounting_n_tolerance(n_all):
            continue

        signs = {
            all_s: 1.0,
            a_s: -1.0,
            b_s: -1.0,
        }

        row = np.zeros(n_vars, dtype=np.float64)
        known_sum = 0.0
        vars_in_equation = []

        for subgroup, sign in signs.items():
            if subgroup in subgroup_to_var:
                var_idx = subgroup_to_var[subgroup]
                row[var_idx] += sign
                vars_in_equation.append(var_idx)
            else:
                known_sum += sign * subgroup_known_count(subgroup)

        if not vars_in_equation:
            continue

        a_rows.append(sqrt_equation * row)
        b_values.append(sqrt_equation * (-known_sum))

        eq_count_used += 1
        touched_vars.update(vars_in_equation)

    if eq_count_used == 0 or not touched_vars:
        return pred_pct, touched, eq_count_used

    A = np.vstack(a_rows)
    b = np.asarray(b_values, dtype=np.float64)

    lower = np.zeros(n_vars, dtype=np.float64)
    upper = np.asarray(var_n, dtype=np.float64)

    try:
        if HAVE_ACCOUNTING_LSQ_LINEAR:
            solution = lsq_linear(
                A,
                b,
                bounds=(lower, upper),
                method="trf",
                lsmr_tol="auto",
                max_iter=100,
            )

            solved_counts = solution.x

        else:
            solved_counts, *_ = np.linalg.lstsq(A, b, rcond=None)
            solved_counts = np.clip(solved_counts, lower, upper)

    except Exception:
        solved_counts, *_ = np.linalg.lstsq(A, b, rcond=None)
        solved_counts = np.clip(solved_counts, lower, upper)

    for var_idx in touched_vars:
        local_idx = var_to_local[var_idx]
        n_students = var_n[var_idx]

        if np.isfinite(n_students) and n_students > 0:
            pred_value = 100.0 * float(solved_counts[var_idx]) / n_students
            pred_pct[local_idx] = np.float32(np.clip(pred_value, 0, 100))
            touched[local_idx] = True

    return pred_pct, touched, eq_count_used


def solve_many_accounting_groups(
    query_df,
    known_lookup,
    prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    """
    Solve all accounting groups for one validation or test frame.
    """
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))

    prior_pct = np.asarray(prior_pct, dtype=np.float32).reshape(-1)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)
    eq_counts = np.zeros(len(query_df), dtype=np.float32)

    for group_key, group_df in query_df.groupby("group_key", sort=False):
        local_positions = group_df["local_pos"].to_numpy(dtype=np.int64)

        known_group = known_lookup.get(str(group_key), {})

        group_pred, group_touched, group_eq_count = solve_one_accounting_group(
            query_group_df=group_df.reset_index(drop=True),
            known_group=known_group,
            prior_pct=prior_pct[local_positions],
            identities=identities,
            equation_weight=equation_weight,
            prior_weight=prior_weight,
        )

        pred[local_positions] = group_pred
        touched[local_positions] = group_touched
        eq_counts[local_positions] = group_eq_count

    return pred, touched, eq_counts

def run_accounting_solver_for_prior(
    prior_name,
    prior_display_name,
    base_oof,
    base_test,
    equation_weight_grid,
    lambda_grid,
    prior_weight=1.0,
):
    """
    Run the accounting solver for one prior model and save OOF/test artifacts.
    """
    print("\n" + "=" * 100)
    print("Accounting solver prior:", prior_display_name)
    print("=" * 100)

    base_oof = np.asarray(base_oof, dtype=np.float32).reshape(-1)
    base_test = np.asarray(base_test, dtype=np.float32).reshape(-1)

    if len(base_oof) != n_train_accounting:
        raise ValueError(f"OOF prior length mismatch for {prior_name}.")

    if len(base_test) != n_test_accounting:
        raise ValueError(f"Test prior length mismatch for {prior_name}.")

    base_oof_mse = float(mean_squared_error(y_accounting, base_oof))

    print("Base OOF MSE:", base_oof_mse)

    folds = list(
        KFold(
            n_splits=ACCOUNTING_N_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE,
        ).split(np.arange(n_train_accounting))
    )

    screen_rows = []
    solver_oof_by_equation_weight = {}
    touched_oof_by_equation_weight = {}

    for equation_weight in equation_weight_grid:
        equation_weight = float(equation_weight)

        solver_oof = np.full(n_train_accounting, np.nan, dtype=np.float32)
        touched_oof = np.zeros(n_train_accounting, dtype=bool)

        print(f"\nEquation weight = {equation_weight:g}")

        for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
            known_lookup_fold = build_accounting_known_lookup(
                train_accounting_df,
                train_idx,
            )

            query_fold = train_accounting_df.iloc[valid_idx].reset_index(drop=True)
            prior_fold = base_oof[valid_idx]

            fold_solver_pred, fold_touched, fold_eq_counts = solve_many_accounting_groups(
                query_df=query_fold,
                known_lookup=known_lookup_fold,
                prior_pct=prior_fold,
                identities=accounting_identities,
                equation_weight=equation_weight,
                prior_weight=prior_weight,
            )

            solver_oof[valid_idx] = fold_solver_pred
            touched_oof[valid_idx] = fold_touched

            print(
                f"  fold {fold_id}: covered {int(fold_touched.sum())} / {len(valid_idx)}"
            )

        solver_oof_by_equation_weight[equation_weight] = solver_oof
        touched_oof_by_equation_weight[equation_weight] = touched_oof

        covered_oof = np.isfinite(solver_oof) & touched_oof

        print(f"  total OOF covered: {int(covered_oof.sum())} / {n_train_accounting}")

        if int(covered_oof.sum()) == 0:
            continue

        for lambda_solver in lambda_grid:
            blended_oof = base_oof.copy()

            blended_oof[covered_oof] = (
                base_oof[covered_oof]
                + float(lambda_solver) * (
                    solver_oof[covered_oof] - base_oof[covered_oof]
                )
            )

            blended_oof = np.clip(blended_oof, 0, 100).astype(np.float32)

            oof_mse = float(mean_squared_error(y_accounting, blended_oof))

            screen_rows.append({
                "prior_name": prior_name,
                "prior_display_name": prior_display_name,
                "equation_weight": equation_weight,
                "prior_weight": float(prior_weight),
                "lambda_solver": float(lambda_solver),
                "covered_rows": int(covered_oof.sum()),
                "coverage_rate": float(covered_oof.mean()),
                "base_oof_mse": base_oof_mse,
                "oof_mse": oof_mse,
                "gain_vs_base": base_oof_mse - oof_mse,
            })

    if not screen_rows:
        raise ValueError(f"No accounting solver rows were covered for {prior_name}.")

    screen = (
        pd.DataFrame(screen_rows)
        .sort_values("oof_mse")
        .reset_index(drop=True)
    )

    best = screen.iloc[0].copy()

    best_equation_weight = float(best["equation_weight"])
    best_lambda_solver = float(best["lambda_solver"])

    best_solver_oof = solver_oof_by_equation_weight[best_equation_weight]
    best_touched_oof = touched_oof_by_equation_weight[best_equation_weight]
    best_covered_oof = np.isfinite(best_solver_oof) & best_touched_oof

    best_oof = base_oof.copy()

    best_oof[best_covered_oof] = (
        base_oof[best_covered_oof]
        + best_lambda_solver * (
            best_solver_oof[best_covered_oof] - base_oof[best_covered_oof]
        )
    )

    best_oof = np.clip(best_oof, 0, 100).astype(np.float32)
    best_oof_mse = float(mean_squared_error(y_accounting, best_oof))

    known_lookup_full = build_accounting_known_lookup(
        train_accounting_df,
        np.arange(n_train_accounting),
    )

    solver_test, touched_test, eq_counts_test = solve_many_accounting_groups(
        query_df=test_accounting_df.reset_index(drop=True),
        known_lookup=known_lookup_full,
        prior_pct=base_test,
        identities=accounting_identities,
        equation_weight=best_equation_weight,
        prior_weight=prior_weight,
    )

    covered_test = np.isfinite(solver_test) & touched_test

    best_test = base_test.copy()

    best_test[covered_test] = (
        base_test[covered_test]
        + best_lambda_solver * (
            solver_test[covered_test] - base_test[covered_test]
        )
    )

    best_test = np.clip(best_test, 0, 100).astype(np.float32)

    screen_path = RESULTS_DIR / f"accounting_solver_{prior_name}_screen.csv"
    oof_path = RESULTS_DIR / f"oof_accounting_solver_{prior_name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_accounting_solver_{prior_name}.csv"

    base_submission_path = Path(f"submission_base_{prior_name}.csv")
    accounting_submission_path = Path(f"submission_accounting_solver_{prior_name}.csv")

    screen.to_csv(screen_path, index=False)

    pd.DataFrame({
        "row_index": np.arange(n_train_accounting),
        TARGET_COL: y_accounting,
        "prior_name": prior_name,
        "base_pred": base_oof,
        "solver_pred": best_solver_oof,
        "solver_covered": best_covered_oof.astype(int),
        "equation_weight": best_equation_weight,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda_solver,
        "pred_clipped": best_oof,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_output,
        "prior_name": prior_name,
        "base_pred": base_test,
        "solver_pred": solver_test,
        "solver_covered": covered_test.astype(int),
        "equation_weight": best_equation_weight,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda_solver,
        TARGET_COL: best_test,
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_output,
        TARGET_COL: np.clip(base_test, 0, 100),
    }).to_csv(base_submission_path, index=False)

    final_submission = pd.DataFrame({
        ID_COL: test_ids_output,
        TARGET_COL: best_test,
    })

    final_submission.to_csv(accounting_submission_path, index=False)

    if final_submission.shape != (n_test_accounting, 2):
        raise ValueError(f"Submission shape is wrong for {prior_name}: {final_submission.shape}")

    if list(final_submission.columns) != [ID_COL, TARGET_COL]:
        raise ValueError(f"Submission columns are wrong for {prior_name}: {list(final_submission.columns)}")

    if final_submission[ID_COL].isna().any():
        raise ValueError(f"Missing IDs in submission for {prior_name}.")

    if final_submission[TARGET_COL].isna().any():
        raise ValueError(f"Missing predictions in submission for {prior_name}.")

    if not np.isfinite(final_submission[TARGET_COL]).all():
        raise ValueError(f"Non-finite predictions in submission for {prior_name}.")

    if not final_submission[TARGET_COL].between(0, 100).all():
        raise ValueError(f"Predictions outside [0, 100] in submission for {prior_name}.")

    summary = {
        "prior_name": prior_name,
        "prior_display_name": prior_display_name,
        "base_oof_mse": base_oof_mse,
        "best_accounting_oof_mse": best_oof_mse,
        "gain_vs_base": base_oof_mse - best_oof_mse,
        "equation_weight": best_equation_weight,
        "prior_weight": float(prior_weight),
        "lambda_solver": best_lambda_solver,
        "oof_covered_rows": int(best_covered_oof.sum()),
        "oof_coverage_rate": float(best_covered_oof.mean()),
        "test_covered_rows": int(covered_test.sum()),
        "test_coverage_rate": float(covered_test.mean()),
        "base_submission_path": str(base_submission_path),
        "accounting_submission_path": str(accounting_submission_path),
        "screen_path": str(screen_path),
        "oof_path": str(oof_path),
        "testpred_path": str(testpred_path),
    }

    print("\nBest accounting config:")
    print(pd.Series(summary).to_string())

    print("\nSaved:")
    print("-", base_submission_path)
    print("-", accounting_submission_path)
    print("-", screen_path)
    print("-", oof_path)
    print("-", testpred_path)

    gc.collect()

    return summary, screen


ACCOUNTING_EQUATION_WEIGHT_GRID = [100.0, 1000.0, 10000.0, 100000.0]

ACCOUNTING_LAMBDA_GRID = np.unique(
    np.concatenate([
        np.linspace(-0.50, 1.50, 501),
        np.array([0.0, 0.25, 0.50, 0.75, 0.916, 0.924, 0.932, 0.992, 1.0]),
    ])
)

FINAL_ACCOUNTING_PRIOR_NAME = "convex_te_blend_prior"
prior_data = accounting_prior_data[FINAL_ACCOUNTING_PRIOR_NAME]

summary, screen = run_accounting_solver_for_prior(
    prior_name=FINAL_ACCOUNTING_PRIOR_NAME,
    prior_display_name=prior_data["display_name"],
    base_oof=prior_data["oof_pred"],
    base_test=prior_data["test_pred"],
    equation_weight_grid=ACCOUNTING_EQUATION_WEIGHT_GRID,
    lambda_grid=ACCOUNTING_LAMBDA_GRID,
    prior_weight=1.0,
)

accounting_solver_summary = pd.DataFrame([summary])
accounting_solver_full_screen = screen.sort_values("oof_mse").reset_index(drop=True)

ACCOUNTING_SOLVER_SUMMARY_PATH = RESULTS_DIR / "accounting_solver_prior_comparison_summary.csv"
ACCOUNTING_SOLVER_FULL_SCREEN_PATH = RESULTS_DIR / "accounting_solver_prior_comparison_full_screen.csv"

accounting_solver_summary.to_csv(ACCOUNTING_SOLVER_SUMMARY_PATH, index=False)
accounting_solver_full_screen.to_csv(ACCOUNTING_SOLVER_FULL_SCREEN_PATH, index=False)

final_accounting_row = accounting_solver_summary.iloc[0]

FINAL_ACCOUNTING_SUBMISSION_PATH = Path(final_accounting_row["accounting_submission_path"])
FINAL_ACCOUNTING_OOF_PATH = Path(final_accounting_row["oof_path"])
FINAL_ACCOUNTING_TESTPRED_PATH = Path(final_accounting_row["testpred_path"])

print("\nFinal accounting model:")
print(final_accounting_row.to_string())
print("\nTop 10 lambda/equation-weight rows:")
print(accounting_solver_full_screen.head(10).to_string(index=False))


final_accounting_oof = pd.read_csv(FINAL_ACCOUNTING_OOF_PATH)
final_accounting_testpred = pd.read_csv(FINAL_ACCOUNTING_TESTPRED_PATH)
final_accounting_submission = pd.read_csv(FINAL_ACCOUNTING_SUBMISSION_PATH)

if "row_index" in final_accounting_oof.columns:
    final_accounting_oof = (
        final_accounting_oof
        .sort_values("row_index")
        .reset_index(drop=True)
    )

final_accounting_oof_pred = final_accounting_oof["pred_clipped"].to_numpy(dtype=float)

final_accounting_oof_mse = mean_squared_error(
    y_accounting,
    final_accounting_oof_pred,
)

final_accounting_base_oof_mse = float(final_accounting_row["base_oof_mse"])
final_accounting_gain = final_accounting_base_oof_mse - final_accounting_oof_mse

ACCOUNTING_SOLVER_FINAL_PASS = bool(
    FINAL_ACCOUNTING_SUBMISSION_PATH.exists()
    and final_accounting_submission.shape == (n_test_accounting, 2)
    and list(final_accounting_submission.columns) == [ID_COL, TARGET_COL]
    and final_accounting_submission[ID_COL].notna().all()
    and final_accounting_submission[TARGET_COL].notna().all()
    and np.isfinite(final_accounting_submission[TARGET_COL]).all()
    and final_accounting_submission[TARGET_COL].between(0, 100).all()
    and final_accounting_oof_mse < final_accounting_base_oof_mse
)

print("ACCOUNTING_SOLVER_FINAL_PASS:", ACCOUNTING_SOLVER_FINAL_PASS)
print("Final accounting prior:", FINAL_ACCOUNTING_PRIOR_NAME)
print("Final accounting submission:", FINAL_ACCOUNTING_SUBMISSION_PATH)
print("Base prior OOF MSE:", final_accounting_base_oof_mse)
print("Accounting solver OOF MSE:", final_accounting_oof_mse)
print("Gain vs prior:", final_accounting_gain)

print("\nFinal submission prediction summary:")
print(final_accounting_submission[TARGET_COL].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).to_string())

accounting_solver_diagnostic = pd.DataFrame([
    {
        "check": "Accounting priors loaded",
        "value": ACCOUNTING_PRIORS_READY,
        "expected": True,
        "pass": bool(ACCOUNTING_PRIORS_READY),
    },
    {
        "check": "Final prior name",
        "value": FINAL_ACCOUNTING_PRIOR_NAME,
        "expected": "convex_te_blend_prior",
        "pass": bool(FINAL_ACCOUNTING_PRIOR_NAME == "convex_te_blend_prior"),
    },
    {
        "check": "Convex prior base OOF MSE",
        "value": final_accounting_base_oof_mse,
        "expected": "about 78.0715",
        "pass": bool(abs(final_accounting_base_oof_mse - 78.0715) < 0.10),
    },
    {
        "check": "Final accounting OOF MSE",
        "value": final_accounting_oof_mse,
        "expected": "about 54.2930",
        "pass": bool(final_accounting_oof_mse < 54.40),
    },
    {
        "check": "Equation weight",
        "value": float(final_accounting_row["equation_weight"]),
        "expected": 100000.0,
        "pass": bool(float(final_accounting_row["equation_weight"]) == 100000.0),
    },
    {
        "check": "Solver shrinkage",
        "value": float(final_accounting_row["lambda_solver"]),
        "expected": 0.932,
        "pass": bool(abs(float(final_accounting_row["lambda_solver"]) - 0.932) < 1e-9),
    },
    {
        "check": "OOF covered rows",
        "value": int(final_accounting_row["oof_covered_rows"]),
        "expected": 81593,
        "pass": bool(int(final_accounting_row["oof_covered_rows"]) == 81593),
    },
    {
        "check": "Test covered rows",
        "value": int(final_accounting_row["test_covered_rows"]),
        "expected": 45105,
        "pass": bool(int(final_accounting_row["test_covered_rows"]) == 45105),
    },
    {
        "check": "Final submission rows",
        "value": len(final_accounting_submission),
        "expected": n_test_accounting,
        "pass": bool(len(final_accounting_submission) == n_test_accounting),
    },
    {
        "check": "Final submission valid",
        "value": ACCOUNTING_SOLVER_FINAL_PASS,
        "expected": True,
        "pass": bool(ACCOUNTING_SOLVER_FINAL_PASS),
    },
])

accounting_solver_diagnostic

FINAL_REPORT_MODEL_NAME = "accounting_solver_convex_te_blend_prior"

FINAL_REPORT_SUBMISSION_SOURCE = Path("submission_accounting_solver_convex_te_blend_prior.csv")
FINAL_REPORT_SUBMISSION_PATH = Path("submission_final_accounting_solver_convex_te_blend_prior.csv")

if not FINAL_REPORT_SUBMISSION_SOURCE.exists():
    raise FileNotFoundError(
        f"Expected final accounting submission not found: {FINAL_REPORT_SUBMISSION_SOURCE}"
    )

final_report_submission = pd.read_csv(FINAL_REPORT_SUBMISSION_SOURCE)

if final_report_submission.shape != (n_test_accounting, 2):
    raise ValueError(
        f"Final submission has wrong shape: {final_report_submission.shape}"
    )

if list(final_report_submission.columns) != [ID_COL, TARGET_COL]:
    raise ValueError(
        f"Final submission has wrong columns: {list(final_report_submission.columns)}"
    )

if final_report_submission[ID_COL].isna().any():
    raise ValueError("Final submission contains missing assessment IDs.")

if final_report_submission[TARGET_COL].isna().any():
    raise ValueError("Final submission contains missing predictions.")

if not np.isfinite(final_report_submission[TARGET_COL]).all():
    raise ValueError("Final submission contains non-finite predictions.")

if not final_report_submission[TARGET_COL].between(0, 100).all():
    raise ValueError("Final submission contains predictions outside [0, 100].")

final_report_submission.to_csv(
    FINAL_REPORT_SUBMISSION_PATH,
    index=False,
)

FINAL_REPORT_SUBMISSION_READY = True

print("FINAL_REPORT_SUBMISSION_READY:", FINAL_REPORT_SUBMISSION_READY)
print("Final report model:", FINAL_REPORT_MODEL_NAME)
print("Source submission:", FINAL_REPORT_SUBMISSION_SOURCE)
print("Final named submission copy:", FINAL_REPORT_SUBMISSION_PATH)

final_report_submission.head()

# Plain-name alias matching the model name in the report prompt.
FINAL_REPORT_ALIAS_PATH = Path("accounting_solver_convex_te_blend_prior.csv")
final_report_submission.to_csv(FINAL_REPORT_ALIAS_PATH, index=False)
print("Plain-name alias:", FINAL_REPORT_ALIAS_PATH)

Have scipy.optimize.lsq_linear: True
Training rows: 144921
Test rows: 48307

Accounting identities used:
All Students = Female + Male
All Students = Economically Disadvantaged + Not Economically Disadvantaged
ACCOUNTING_PRIORS_READY: True

Accounting solver prior: Verified convex ensemble prior
Base OOF MSE: 78.07151040664739

Equation weight = 100
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 1000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 10000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation we

## Final outputs

After the last cell runs, the submission is available under three names:

- `submission_accounting_solver_convex_te_blend_prior.csv` — source file written by the accounting solver.
- `submission_final_accounting_solver_convex_te_blend_prior.csv` — final named copy from the original notebook.
- `accounting_solver_convex_te_blend_prior.csv` — plain-name alias matching the final model name.
